# Task 2 — Validation of Temporal Knowledge Graphs (Google Colab / Jupyter)

Этот блокнот является **естественным продолжением Task 1**.

Он сначала ищет **локальный исправленный репозиторий рядом с собой** и только если не находит его — использует `git clone` официального GitHub-репозитория.

Эксперт загружает YAML-файл, созданный в первом блокноте, после чего блокнот автоматически:
- строит **эталонный (gold) граф**, который точно воспроизводит ручную reasoning-схему из YAML;
- при желании строит **автоматический temporal KG** по тем же статьям через пайплайн репозитория;
- сохраняет обе версии графа, таблицы триплетов, comparison summary и визуализации;
- показывает эксперту **текстовое представление триплетов** и **интерактивные визуализации** для ручной валидации.

Обновление: notebook рассчитан на универсальный онлайн-резолв статей из `papers[]` и `steps[].sources[]` (DOI, landing pages, direct PDF, source hints).

Если рядом лежит уже исправленный архив репозитория — блокнот использует его как есть. Если найден старый clone без поддержки `interval`, блокнот автоматически применяет минимальный compatibility patch для Task 2 перед импортом модулей.


# Пошаговый туториал

## Шаг 1. Подготовьте репозиторий
Рекомендуемый вариант — распаковать архив `top-papers-graph-fixed-v5.zip` в `/content` или рядом с ноутбуком.

## Шаг 2. Запустите ячейку «Установка и импорт»
Она:
- найдёт локальный репозиторий автоматически;
- только если локальный репозиторий не найден, клонирует официальный GitHub-репозиторий;
- при необходимости автоматически пропатчит `schemas.py`, чтобы Task 2 принимал `interval`, legacy-алиасы (`range`, `period`, `timespan`) и мягко приводил нестроковые `subject/predicate/object` к строкам до валидации;
- установит зависимости для Task 2 notebook;
- добавит локальный `src` в `sys.path` как надёжный fallback;
- импортирует функции Task 2.

## Шаг 3. Запустите ячейку «Вспомогательные функции»
Она подготовит UI, рендеринг таблиц и графов.

## Шаг 4. Запустите ячейку «Форма Task 2»
В ней нужно:
1. загрузить YAML из Task 1;
2. выбрать, строить ли автоматический граф;
3. при желании нажать кнопку **«Проверить форму»**, чтобы убедиться, что YAML считался корректно.

## Шаг 5. Запустите ячейку «Запуск Task 2 из отдельной ячейки»
Это важное изменение для Google Colab и части Jupyter-окружений:
- сам pipeline теперь запускается **из отдельной исполняемой ячейки**, а не из `ipywidgets` callback;
- поэтому notebook остаётся в состоянии **Running**, пока строится bundle;
- из-за этого Colab не воспринимает долгий Task 2 как «все ячейки уже выполнены и notebook простаивает».

Обновление: после сборки bundle доступны вкладки просмотра эталонного и автоматически сгенерированного графов, а также интерфейс экспертной валидации с экспортом результатов.

In [1]:
# @title
# Установка и импорт (запустите эту ячейку)
import gc, json, os, sys, tempfile, subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def run_optional(cmd, cwd=None, label=None):
    try:
        run(cmd, cwd=cwd)
        return True
    except Exception as e:
        label = label or ' '.join(map(str, cmd))
        print(f'[warn] optional install failed: {label}: {type(e).__name__}: {e}')
        return False


def find_repo_root() -> Path | None:
    candidates = []
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    search_bases = [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]
    patterns = (
        'top-papers-graph-fixed-v5',
        'top-papers-graph-fixed-v4',
        'top-papers-graph-fixed-v3',
        'top-papers-graph-fixed*',
        'top-papers-graph-main*',
        'top-papers-graph',
    )
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {SRC_DIR}')



COMPAT_SCHEMAS_TEXT = r'''from __future__ import annotations

from hashlib import sha1
from typing import Any, Literal, Optional

from pydantic import BaseModel, Field, model_validator


Granularity = Literal["year", "month", "day", "interval"]


def _infer_granularity_from_value(value: Any) -> str:
    text = str(value or "").strip()
    if not text:
        return "year"
    if text in {"unknown", "+inf", "-inf"}:
        return "year"
    if len(text) >= 10 and text[4:5] == "-" and text[7:8] == "-":
        return "day"
    if len(text) >= 7 and text[4:5] == "-":
        return "month"
    return "year"


def normalize_granularity(value: Any, *, start: Any = None, end: Any = None, default: str = "year") -> str:
    text = str(value or "").strip().lower()
    aliases = {
        "": default,
        "unknown": default,
        "year": "year",
        "annual": "year",
        "month": "month",
        "monthly": "month",
        "day": "day",
        "daily": "day",
        "interval": "interval",
        "range": "interval",
        "period": "interval",
        "timespan": "interval",
        "time_span": "interval",
        "date_range": "interval",
    }
    if text in aliases:
        return aliases[text]
    inferred_start = _infer_granularity_from_value(start)
    inferred_end = _infer_granularity_from_value(end)
    if start not in (None, "") and end not in (None, "") and inferred_start == inferred_end:
        return "interval"
    if start not in (None, "") or end not in (None, ""):
        return "interval"
    return default


def _coerce_relaxed_string(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, bytes):
        try:
            return value.decode("utf-8", "ignore").strip()
        except Exception:
            return str(value).strip()
    if isinstance(value, (int, float, bool)):
        return str(value)
    if isinstance(value, dict):
        for key in ("text", "label", "name", "title", "value", "object", "entity", "term", "mention", "id"):
            if key in value:
                text = _coerce_relaxed_string(value.get(key))
                if text:
                    return text
        parts: list[str] = []
        for key, item in value.items():
            text = _coerce_relaxed_string(item)
            if text:
                parts.append(f"{key}={text}")
        return " | ".join(parts)
    if isinstance(value, (list, tuple, set)):
        parts: list[str] = []
        for item in value:
            text = _coerce_relaxed_string(item)
            if text:
                parts.append(text)
        return " | ".join(parts)
    return str(value).strip()


EventSplit = Literal["train", "valid", "test"]
EventType = Literal["extracted", "reviewed", "corrected"]


class TimeInterval(BaseModel):
    start: Optional[str] = Field(default=None, description="ISO дата или год: YYYY / YYYY-MM / YYYY-MM-DD")
    end: Optional[str] = Field(default=None, description="ISO дата/год конца интервала (включительно)")
    granularity: Granularity = "year"

    @model_validator(mode="before")
    @classmethod
    def _normalize_input(cls, data: Any) -> Any:
        if not isinstance(data, dict):
            return data
        payload = dict(data)
        if payload.get("start") is not None:
            payload["start"] = _coerce_relaxed_string(payload.get("start"))
        if payload.get("end") is not None:
            payload["end"] = _coerce_relaxed_string(payload.get("end"))
        payload["granularity"] = normalize_granularity(
            payload.get("granularity"),
            start=payload.get("start"),
            end=payload.get("end"),
            default="year",
        )
        return payload

    def key(self) -> str:
        start = self.start or ""
        end = self.end or start
        return f"{self.granularity}:{start}:{end}"


class TemporalTriplet(BaseModel):
    subject: str
    predicate: str
    object: str
    confidence: float = Field(ge=0.0, le=1.0, default=0.6)
    polarity: Literal["supports", "contradicts", "unknown"] = "unknown"
    time: Optional[TimeInterval] = None
    time_source: Literal["extracted", "paper_year_fallback"] = "extracted"
    evidence_quote: Optional[str] = Field(
        default=None,
        description="Короткая цитата/фрагмент (<=200 символов) из исходного текста, подтверждающий триплет.",
    )

    @model_validator(mode="before")
    @classmethod
    def _normalize_input(cls, data: Any) -> Any:
        if not isinstance(data, dict):
            return data
        payload = dict(data)
        payload["subject"] = _coerce_relaxed_string(payload.get("subject"))
        payload["predicate"] = _coerce_relaxed_string(payload.get("predicate"))
        payload["object"] = _coerce_relaxed_string(payload.get("object"))
        if payload.get("evidence_quote") is not None:
            payload["evidence_quote"] = _coerce_relaxed_string(payload.get("evidence_quote"))
        time_payload = payload.get("time")
        if isinstance(time_payload, dict):
            time_payload = dict(time_payload)
            if time_payload.get("start") is not None:
                time_payload["start"] = _coerce_relaxed_string(time_payload.get("start"))
            if time_payload.get("end") is not None:
                time_payload["end"] = _coerce_relaxed_string(time_payload.get("end"))
            time_payload["granularity"] = normalize_granularity(
                time_payload.get("granularity"),
                start=time_payload.get("start"),
                end=time_payload.get("end"),
                default="year",
            )
            payload["time"] = time_payload
        return payload

    def as_text(self) -> str:
        time_part = ""
        if self.time is not None:
            time_part = f" | time={self.time.key()}"
        quote_part = f" | evidence={self.evidence_quote}" if self.evidence_quote else ""
        return f"{self.subject} | {self.predicate} | {self.object}{time_part}{quote_part}"


class TemporalEvent(BaseModel):
    event_id: Optional[str] = None
    paper_id: str
    chunk_id: Optional[str] = None
    assertion_id: Optional[str] = None
    subject: str
    predicate: str
    object: str
    ts_start: Optional[str] = None
    ts_end: Optional[str] = None
    granularity: Granularity = "year"

    @model_validator(mode="before")
    @classmethod
    def _normalize_input(cls, data: Any) -> Any:
        if not isinstance(data, dict):
            return data
        payload = dict(data)
        for key in ("event_id", "paper_id", "chunk_id", "assertion_id", "subject", "predicate", "object", "ts_start", "ts_end", "evidence_quote"):
            if payload.get(key) is not None:
                payload[key] = _coerce_relaxed_string(payload.get(key))
        payload["granularity"] = normalize_granularity(
            payload.get("granularity"),
            start=payload.get("ts_start"),
            end=payload.get("ts_end"),
            default="year",
        )
        return payload

    confidence: float = Field(ge=0.0, le=1.0, default=0.6)
    polarity: Literal["supports", "contradicts", "unknown"] = "unknown"
    split: Optional[EventSplit] = None
    event_type: EventType = "extracted"
    extraction_method: str = "llm_triplet"
    weight: float = 1.0
    evidence_quote: Optional[str] = None

    def time_key(self) -> str:
        start = self.ts_start or ""
        end = self.ts_end or start
        return f"{self.granularity}:{start}:{end}"

    def pair_key(self) -> tuple[str, str, str]:
        return (self.subject.strip().lower(), self.predicate.strip().lower(), self.object.strip().lower())

    def sort_key(self) -> tuple[str, str, str, str]:
        return (
            self.ts_start or "",
            self.ts_end or "",
            self.subject.strip().lower(),
            self.object.strip().lower(),
        )

    def as_text(self) -> str:
        quote_part = f" | evidence={self.evidence_quote}" if self.evidence_quote else ""
        return (
            f"{self.subject} | {self.predicate} | {self.object}"
            f" | time={self.time_key()} | conf={self.confidence:.3f}{quote_part}"
        )

    def stable_id(self) -> str:
        if self.event_id:
            return self.event_id
        raw = (
            f"{self.paper_id}|{self.chunk_id or ''}|{self.subject}|{self.predicate}|{self.object}|"
            f"{self.ts_start or ''}|{self.ts_end or ''}|{self.granularity}|{self.event_type}"
        )
        return sha1(raw.encode("utf-8")).hexdigest()[:16]
'''

def ensure_task2_temporal_compat(repo_dir: Path) -> bool:
    """Patch old repository checkouts so Task 2 accepts interval granularity."""
    schemas_path = repo_dir / 'src' / 'scireason' / 'temporal' / 'schemas.py'
    if not schemas_path.exists():
        return False
    try:
        current = schemas_path.read_text(encoding='utf-8')
    except Exception:
        current = ''
    needs_patch = (
        'normalize_granularity' not in current
        or 'Literal["year", "month", "day", "interval"]' not in current
        or 'model_validator' not in current
        or 'def _coerce_relaxed_string' not in current
    )
    if not needs_patch:
        print('Task2 temporal compatibility patch: not needed')
        return False
    schemas_path.write_text(COMPAT_SCHEMAS_TEXT, encoding='utf-8')
    print('Task2 temporal compatibility patch: applied to', schemas_path)
    return True

ensure_task2_temporal_compat(REPO_DIR)


# Стабилизируем поведение PaddleX/PaddleOCR в Colab и локально.
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')
os.environ.setdefault('PADDLE_PDX_MODEL_SOURCE', 'BOS')
os.environ.setdefault('PADDLEOCR_WORKER_TIMEOUT_SECONDS', '90')
os.environ.setdefault('SCIREASON_LOCAL_VLM_MODE', 'worker')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def ensure_paddle_stack():
    """Best-effort install for PaddleOCR doc-parser stack."""
    import importlib.util

    missing = [
        name for name in ('paddle', 'paddleocr', 'langchain_text_splitters')
        if importlib.util.find_spec(name) is None
    ]
    if not missing:
        print('PaddleOCR doc-parser stack: installed')
        return True

    print('PaddleOCR doc-parser stack missing ->', ', '.join(missing))

    ok = run_optional([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/',
        'paddlepaddle==3.2.0'
    ], label='paddlepaddle==3.2.0 (cpu)')
    ok = run_optional([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'paddleocr[doc-parser]>=3.0.0', 'langchain-community>=0.3', 'langchain-text-splitters>=0.3'
    ], label='paddleocr[doc-parser] + langchain splitters') and ok
    if not ok:
        print('[warn] PaddleOCR stack не установлен полностью. Ноутбук продолжит работу; multimodal/OCR шаг можно отключить или доустановить позже.')
    return ok


def _run_isolated_python(code: str, *args: str, timeout: int = 180, extra_env: dict | None = None):
    env = os.environ.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    return subprocess.run(
        [sys.executable, '-c', code, *[str(a) for a in args]],
        capture_output=True,
        text=True,
        timeout=timeout,
        env=env,
    )


def _local_vlm_runtime_check(model_id: str | None = None):
    """Return (ok, reason) for the local Transformers/VLM runtime.

    Проверка идёт в отдельном python subprocess, чтобы не переимпортировать torch
    в текущем kernel после pip-install. Это устраняет класс ошибок вида
    "function '_has_torch_function' already has a docstring", которые возникают
    при повторном/частично очищенном импорте torch в notebook-сессии.
    """
    probe = r'''import json
import sys

requested = (sys.argv[1] if len(sys.argv) > 1 else '').strip()
result = {"ok": False, "reason": ""}

try:
    import torch  # noqa: F401
    from PIL import Image  # noqa: F401
    from transformers import AutoProcessor  # noqa: F401
except Exception as e:
    result["reason"] = f"{type(e).__name__}: {e}"
    print(json.dumps(result, ensure_ascii=False))
    raise SystemExit(0)

try:
    if 'Qwen3-VL' in requested:
        from transformers import Qwen3VLForConditionalGeneration  # noqa: F401
    elif 'Qwen/Qwen2.5-VL' in requested or 'Qwen2.5-VL' in requested:
        from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
        import qwen_vl_utils  # noqa: F401
    else:
        try:
            from transformers import AutoModelForImageTextToText  # noqa: F401
        except Exception:
            from transformers import AutoModelForVision2Seq  # noqa: F401
except Exception as e:
    result["reason"] = f"{type(e).__name__}: {e}"
    print(json.dumps(result, ensure_ascii=False))
    raise SystemExit(0)

result["ok"] = True
print(json.dumps(result, ensure_ascii=False))
'''
    requested = str(model_id or '').strip()
    try:
        proc = _run_isolated_python(probe, requested, timeout=120)
    except subprocess.TimeoutExpired:
        return False, 'TimeoutExpired: isolated_import_probe'

    lines = (proc.stdout or '').strip().splitlines()
    payload = lines[-1] if lines else ''
    if payload:
        try:
            parsed = json.loads(payload)
            return bool(parsed.get('ok')), str(parsed.get('reason') or '')
        except Exception:
            pass

    stderr_lines = (proc.stderr or '').strip().splitlines()
    reason = stderr_lines[-1] if stderr_lines else f'isolated_probe_exit_{proc.returncode}'
    return False, reason


def ensure_transformers_stack(model_id: str | None = None):
    """Best-effort install for local VLM inference with an isolated runtime check."""
    ok, reason = _local_vlm_runtime_check(model_id)
    if ok:
        print('Local Transformers/VLM stack: installed')
        return True

    print('Local Transformers/VLM stack missing or broken ->', reason)

    import importlib.util

    step_ok = True
    if importlib.util.find_spec('torch') is None:
        step_ok = run_optional([
            sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
            '--index-url', 'https://download.pytorch.org/whl/cpu',
            'torch'
        ], label='torch (cpu)') and step_ok

    # Для Qwen2.5-VL модельная карточка рекомендует свежий transformers из git + accelerate.
    step_ok = run_optional([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'git+https://github.com/huggingface/transformers',
        'accelerate>=0.33', 'qwen-vl-utils>=0.0.14', 'sentencepiece>=0.2',
        'pillow>=10.0', 'safetensors>=0.4.3', 'torchvision'
    ], label='transformers/qwen local VLM stack (git+accelerate; image-text compatible)') and step_ok

    import importlib
    importlib.invalidate_caches()

    ok, reason = _local_vlm_runtime_check(model_id)
    if not ok:
        print('[warn] Local Transformers/VLM runtime всё ещё недоступен ->', reason)
        return False

    print('Local Transformers/VLM stack: ready')
    return step_ok and ok


# 1) Лёгкий UI-стек для notebook. pandas здесь специально НЕ ставим отдельно,
#    чтобы не получить ABI-конфликт с numpy после editable-install репозитория.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ipywidgets', 'pyyaml', 'requests',
    'panel', 'hvplot', 'holoviews', 'bokeh', 'pyvis', 'jupyter_bokeh'
])

# 2) Ставим editable-install максимально надёжно.
#    task2_notebook в исправленном репозитории облегчён и не тянет тяжёлые optional stacks.
editable_ok = False
for spec in (
    f'{REPO_DIR}[task2_notebook]',
    f'{REPO_DIR}[temporal,notebook]',
    str(REPO_DIR),
):
    try:
        run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', spec])
        editable_ok = True
        break
    except Exception as e:
        print(f'[warn] editable install failed for {spec}: {type(e).__name__}: {e}')

if not editable_ok:
    # Крайний fallback: регистрируем пакет без зависимостей и ставим минимальный runtime отдельно.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', '-e', str(REPO_DIR)])
    run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'pydantic>=2.6', 'pydantic-settings>=2.2', 'typer>=0.12', 'rich>=13.7',
        'httpx>=0.27', 'tenacity>=8.2', 'python-dotenv>=1.0', 'pyyaml>=6.0',
        'networkx>=3.3', 'numpy>=1.26.4,<2', 'pypdf>=4.0', 'litellm>=1.0',
        'langchain-core>=0.2', 'langgraph>=0.1'
    ])

# 3) Опциональные рантаймы ставим отдельно и без падения всей ячейки.
run_optional([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'g4f>=7.1.2'
], label='g4f>=7.1.2')
TASK2_LOCAL_VLM_RUNTIME_OK = ensure_transformers_stack()
TASK2_LOCAL_VLM_RUNTIME_REASON = _local_vlm_runtime_check()[1] if not TASK2_LOCAL_VLM_RUNTIME_OK else ''
ensure_paddle_stack()

# 4) В Colab часто остаётся предустановленный pandas, собранный под другой ABI NumPy.
#    Принудительно выравниваем совместимую пару после editable-install.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir',
    'numpy>=1.26.4,<2', 'pandas>=2.2.2,<2.4'
])

# 5) Даже если editable-install не прописал import hook, ноутбук всё равно должен работать.
#    Поэтому всегда добавляем локальный src в sys.path как надёжный fallback.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# На случай, если kernel уже видел старые модули
for mod in list(sys.modules):
    if mod == 'numpy' or mod.startswith('numpy.') or mod == 'pandas' or mod.startswith('pandas.'):
        del sys.modules[mod]

import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, HTML, clear_output

try:
    import panel as pn
    pn.extension()
except Exception:
    pn = None

try:
    import hvplot  # noqa: F401
    import hvplot.networkx  # noqa: F401
except Exception:
    pass

try:
    import scireason.task2_validation as _task2_validation_module
    from scireason.task2_validation import (
        build_task2_validation_bundle,
        build_task2_offline_review_package,
        load_task1_yaml,
        make_hvplot_payload,
        get_task2_review_state_paths,
        save_task2_review_state,
        load_task2_review_state,
    )
except Exception as e:
    raise RuntimeError(
        'Не удалось импортировать полноценный модуль scireason.task2_validation. '
        'Ноутбук не будет переключаться на fallback-реализацию Task 2. '
        'Убедитесь, что рядом с ноутбуком лежит исправленный репозиторий, и заново запустите ячейку «Установка и импорт». '
        f'Исходная ошибка: {type(e).__name__}: {e}'
    ) from e


from scireason.config import settings as _repo_settings
from scireason.llm import list_available_g4f_models as _repo_list_available_g4f_models

TASK2_NOTEBOOK_PIPELINE_MODE = 'full'
TASK2_NOTEBOOK_PIPELINE_SOURCE = str(Path(_task2_validation_module.__file__).resolve())
TASK2_DEFAULT_G4F_MODEL = getattr(_repo_settings, 'task2_default_g4f_model', 'auto')
TASK2_DEFAULT_LOCAL_MODEL = getattr(_repo_settings, 'task2_default_local_vlm_model', 'Qwen/Qwen2.5-VL-3B-Instruct')
TASK2_G4F_PROBE_TIMEOUT_SECONDS = int(os.getenv('TASK2_G4F_PROBE_TIMEOUT_SECONDS', '25') or '25')
TASK2_G4F_PROBE_MAX_MODELS = int(os.getenv('TASK2_G4F_PROBE_MAX_MODELS', '0') or '0')
TASK2_LOCAL_VLM_RUNTIME_OK = bool(globals().get('TASK2_LOCAL_VLM_RUNTIME_OK', False))
TASK2_LOCAL_VLM_RUNTIME_REASON = str(globals().get('TASK2_LOCAL_VLM_RUNTIME_REASON', '') or '')
_TASK2_G4F_PROBE_IMAGE_B64 = (
    'iVBORw0KGgoAAAANSUhEUgAAAPAAAAB4CAIAAABD1OhwAAAN/UlEQVR4nO3df1RT9f8H8AtsA4FtDNBxojQiRdIDGWZyQCVOIKKUpWmZGudAR+l0ICuPdjqnY53TD4g8KYl1OifjRwJyRqfTdlYgHTRSG4HAUUlq5GGAxAYbbuP32PeP9eVD4O77vbt7t/H29fhL8X3fLx1E6wYAAEAAp1Ylpr4BAAAAuagBAABQEgAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAA4C1Wr15dpVKp7g9j9+7d8+bNy9HREW9vb6VSKT09/ezZs4sXL77wwgsPPvjg7NmzV69e7d+/v1Kp1H0O27ZtK5VK33333b/++mvPnj3r1q2rU6eOUqlUqVTGxsZ8fHwePXq0b9/+X3/99f3336dSqdWrV+fk5CxZsmT37t29e/fm5+fv27fv448//umnn5YsWfLJJ59cv3598+bNQ4cOTZ06ddu2bQ8++ODs2bP5+flTp06dNWtW+fLlFy1adO7cuYEDB8rLy+fPn5+cnDx58mTz5s3btm0bM2bMli1b2rZtO3bs2MGDBxcvXly4cKFUKeXk5Dx48GDNmjW9e/fu3r37c889N3v27CuuuEKhUAwODo4aNapFixb29vYfffTRnDlz9uzZc+LEiUOHDm3fvv2ZZ55RKpUf/vCHbdu2bd26NT09PXPmzLnd7t27d7du3Zo+ffqGDRvOnDnz+eefL1269PLlyz09PevWrbvzzjuhoaG33npr5syZc+bM2b9//7Fjx0ql0q5du+Li4mJiYlKpVDwePz09fejQocTERE6nMzMzly9fPnHiRO/evXfu3NmxY0dBQYFKpSIiIrq6uvbt29fX17d27drRo0cPHjyYlZW1c+fOwYMHr1692rlz58qVK1OnTg0NDW3YsKG4uHj58uWNjY1KpXLr1q0TJ05MmDBh0aJF6enp27dvz8/P79u377p16xYvXpw/f37btm1RUVH5+fmbN2/+8ssvH3744b59+3bs2DF69OjZs2fPnj279dZbzzzzTH5+/vDhwxMnTnzyySfFxcW1tbV8fHzOnj17+eWXr1+/Xq1We/fuHTZs2IULF8rKytq0aVNpaWl0dHT79u2fffZZhUIhLS2tZ8+eZ8+eLV26dMyYMVOnTqWlpdWqVevUqfPLL79cvXr1/Pnz33777eHDh8ePH2/Xrl1aWprD4fDHH3/csmXLtm3b7r///rVr165YsaK7u3vt2rW2bdvGjh3bvn17eXl5e/fuXbRo0dKlS4uLi8vKyqqqqpYtW+bn58fHx1dXV8fFxZWVlVVVV99993HjxgMHDjQ2Nk6dOjV58uRz587V19fX19ePGzdOTExMS0vr3LmzWq1eu3bt1atX09LS3n333fnz5/fu3Tt69Ohu3brZ2dlr165VKBTbtm0bMmSIl5dXRETEtGnTnnjiiY4dOx4+fDg2NrZ+/fqVK1fK5XKZmZn79++/du1aQ0NDW1tbXFxcnTt3JiYmTp06dciQIf/85z9vvvmmVq0aAIDjP+YoM1oAAADwA7kLQAAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAAIEoAAABECQAA+AtdYkvpBNaLqAAAAABJRU5ErkJggg=='
)
_TASK2_G4F_SCAN_RESULTS = []
_TASK2_G4F_LAST_SCAN_META = {}


def _task2_dedup_keep_order(values):
    seen = set()
    out = []
    for value in values:
        key = str(value or '').strip()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(key)
    return out


def _task2_g4f_provider_name(provider_obj):
    if provider_obj is None:
        return ''
    try:
        if isinstance(provider_obj, (list, tuple, set)):
            names = [_task2_g4f_provider_name(x) for x in provider_obj]
            names = [n for n in names if n]
            return ', '.join(names)
        name = getattr(provider_obj, '__name__', None)
        if name:
            return str(name)
        name = getattr(provider_obj, 'name', None)
        if name:
            return str(name)
        return str(provider_obj)
    except Exception:
        return ''


def _task2_g4f_registry_rows():
    rows = []
    try:
        from g4f import models as gm  # type: ignore
    except Exception:
        return rows

    names = []
    try:
        Model = getattr(gm, 'Model', None)
        if Model is not None and hasattr(Model, '__all__'):
            cand = Model.__all__()  # type: ignore
            if isinstance(cand, (list, tuple)):
                names = list(cand)
    except Exception:
        pass

    if not names:
        try:
            names = list(getattr(gm, '_all_models', []) or [])
        except Exception:
            names = []

    if not names:
        try:
            names = list((getattr(gm, '__models__', {}) or {}).keys())
        except Exception:
            names = []

    if not names:
        try:
            repo_names = _repo_list_available_g4f_models(include_auto=False)
            names = list(repo_names or [])
        except Exception:
            names = []

    names = _task2_dedup_keep_order(names)

    VisionModel = getattr(gm, 'VisionModel', None)
    ImageModel = getattr(gm, 'ImageModel', None)
    AudioModel = getattr(gm, 'AudioModel', None)
    VideoModel = getattr(gm, 'VideoModel', None)

    ModelRegistry = None
    try:
        from g4f.models import ModelRegistry as _G4FModelRegistry  # type: ignore
        ModelRegistry = _G4FModelRegistry
    except Exception:
        ModelRegistry = None

    for name in names:
        model_obj = None
        if ModelRegistry is not None:
            try:
                model_obj = ModelRegistry.get(name)
            except Exception:
                model_obj = None
        is_image = bool(ImageModel is not None and model_obj is not None and isinstance(model_obj, ImageModel))
        is_audio = bool(AudioModel is not None and model_obj is not None and isinstance(model_obj, AudioModel))
        is_video = bool(VideoModel is not None and model_obj is not None and isinstance(model_obj, VideoModel))
        if is_image or is_audio or is_video:
            continue
        is_vision = bool(VisionModel is not None and model_obj is not None and isinstance(model_obj, VisionModel))
        base_provider = str(getattr(model_obj, 'base_provider', '') or '') if model_obj is not None else ''
        best_provider = _task2_g4f_provider_name(getattr(model_obj, 'best_provider', None) if model_obj is not None else None)
        rows.append({
            'name': str(name),
            'kind': 'vision' if is_vision else 'chat',
            'base_provider': base_provider,
            'best_provider': best_provider,
        })

    defaults = [TASK2_DEFAULT_G4F_MODEL, 'gpt-4o', 'gpt-4o-mini', 'gemini-2.5-pro', 'gemini-2.5-flash']
    existing = {row['name'] for row in rows}
    for fallback_name in defaults:
        if fallback_name and fallback_name != 'auto' and fallback_name not in existing:
            rows.append({
                'name': fallback_name,
                'kind': 'chat',
                'base_provider': '',
                'best_provider': '',
            })
    return rows


def _task2_mm_relevance_score(row):
    name = str(row.get('name') or '')
    low = name.lower()
    score = 0

    preferred = [
        'gemini-2.5-pro',
        'gpt-4o',
        'gemini-2.5-flash',
        'gpt-4o-mini',
        'qwen-2.5-vl-72b',
        'qwen-2-vl-72b',
        'qwen-2.5-vl-7b',
        'phi-4-multimodal',
        'llama-3.2-90b',
        'llama-3.2-11b',
        'janus-pro-7b',
        'claude-3.7-sonnet',
        'claude-3.5-sonnet',
        'gemini-2.0-flash',
        'gemini-1.5-pro',
        'gemini-1.5-flash',
        'r1-1776',
        'deepseek-v3',
        'deepseek-r1',
        'qwen-2.5-72b',
        'llama-3.3-70b',
        'gpt-4',
    ]
    if name in preferred:
        score += 2000 - preferred.index(name) * 40

    if row.get('kind') == 'vision':
        score += 900
    if row.get('best_provider'):
        score += 60

    if '4o' in low:
        score += 300
    if 'gemini' in low:
        score += 280
    if 'vision' in low or '-vl' in low or 'multimodal' in low:
        score += 260
    if 'claude-3' in low:
        score += 200
    if 'llava' in low or 'janus' in low:
        score += 160
    if 'phi-4-multimodal' in low:
        score += 180
    if 'r1-1776' in low or 'deepseek-r1' in low:
        score -= 80
    if low.startswith('r1') or 'text-embedding' in low or 'tts' in low:
        score -= 500
    return score


def _task2_rank_g4f_rows(rows):
    ranked = sorted(
        list(rows),
        key=lambda row: (-_task2_mm_relevance_score(row), str(row.get('name') or '').lower())
    )
    return ranked


def list_available_g4f_models(include_auto: bool = True):
    rows = _task2_rank_g4f_rows(_task2_g4f_registry_rows())
    models = [row['name'] for row in rows]
    if include_auto:
        models = ['auto', *models]
    return _task2_dedup_keep_order(models) or (['auto'] if include_auto else [])


def _task2_relevant_g4f_models_from_results(results):
    working_rows = []
    for item in results or []:
        if not item.get('ok'):
            continue
        working_rows.append({
            'name': item.get('model') or '',
            'kind': item.get('kind') or 'chat',
            'base_provider': item.get('base_provider') or '',
            'best_provider': item.get('best_provider') or '',
        })
    ranked = _task2_rank_g4f_rows(working_rows)
    return [row['name'] for row in ranked if row.get('name')]


def _task2_build_g4f_dropdown_options(*, prefer_verified: bool = False, include_auto: bool = True):
    verified = _task2_relevant_g4f_models_from_results(_TASK2_G4F_SCAN_RESULTS)
    candidate_models = verified if (prefer_verified and verified) else list_available_g4f_models(include_auto=False)
    options = []
    if include_auto:
        auto_label = 'auto (repo default router)'
        if verified:
            auto_label = f'auto (verified pool: {len(verified)})'
        options.append((auto_label, 'auto'))
    for name in candidate_models:
        label = name
        match = next((item for item in _TASK2_G4F_SCAN_RESULTS if item.get('model') == name and item.get('ok')), None)
        if match:
            provider_txt = str(match.get('best_provider') or '').strip()
            mode_txt = str(match.get('mode') or '').strip()
            extras = [x for x in [provider_txt, mode_txt] if x]
            if extras:
                label = f"{name} — {' | '.join(extras)}"
            else:
                label = f'{name} — verified'
        options.append((label, name))
    if TASK2_DEFAULT_G4F_MODEL and TASK2_DEFAULT_G4F_MODEL != 'auto' and TASK2_DEFAULT_G4F_MODEL not in [value for _, value in options]:
        options.append((f'{TASK2_DEFAULT_G4F_MODEL} — repo default', TASK2_DEFAULT_G4F_MODEL))
    return options or [('auto (repo default router)', 'auto')]


def _task2_probe_g4f_model(model_name: str, timeout_seconds: int | None = None):
    timeout_seconds = int(timeout_seconds or TASK2_G4F_PROBE_TIMEOUT_SECONDS or 25)
    env = os.environ.copy()
    env['TASK2_G4F_PROBE_IMAGE_B64'] = _TASK2_G4F_PROBE_IMAGE_B64
    env['TASK2_G4F_PROBE_API_KEY'] = str(getattr(_repo_settings, 'g4f_api_key', None) or os.getenv('G4F_API_KEY') or '')
    env['TASK2_G4F_PROBE_PROVIDERS'] = str(getattr(_repo_settings, 'g4f_providers', None) or os.getenv('G4F_PROVIDERS') or '')
    try:
        proc = subprocess.run(
            [sys.executable, '-c', "import json\nimport os\nimport re\nimport sys\n\nB64 = os.environ.get('TASK2_G4F_PROBE_IMAGE_B64', '')\nMODEL = sys.argv[1]\nAPI_KEY = os.environ.get('TASK2_G4F_PROBE_API_KEY') or None\nRAW_PROVIDERS = (os.environ.get('TASK2_G4F_PROBE_PROVIDERS') or '').strip()\n\nproviders = None\nif RAW_PROVIDERS:\n    plist = [p.strip() for p in RAW_PROVIDERS.split(',') if p.strip()]\n    if len(plist) == 1:\n        providers = plist[0]\n    elif plist:\n        providers = plist\n\ndata_url = 'data:image/png;base64,' + B64\nprompt = 'Return the exact uppercase word shown in the image. Reply with one word only.'\n\nresult = {\n    'model': MODEL,\n    'ok': False,\n    'response': '',\n    'mode': '',\n    'reason': '',\n}\n\ntry:\n    from g4f.client import Client as G4FClient  # type: ignore\n    client = G4FClient(api_key=API_KEY)\nexcept Exception as e:\n    result['reason'] = f'g4f_import:{type(e).__name__}:{e}'\n    print(json.dumps(result, ensure_ascii=False))\n    raise SystemExit(0)\n\nattempts = [\n    (\n        'openai_style',\n        lambda: client.chat.completions.create(\n            model=MODEL,\n            provider=providers,\n            messages=[{\n                'role': 'user',\n                'content': [\n                    {'type': 'text', 'text': prompt},\n                    {'type': 'image_url', 'image_url': {'url': data_url}},\n                ],\n            }],\n        ),\n    ),\n    (\n        'images_arg',\n        lambda: client.chat.completions.create(\n            model=MODEL,\n            provider=providers,\n            messages=[{'role': 'user', 'content': prompt}],\n            images=[[data_url, 'graph_probe.png']],\n        ),\n    ),\n]\n\nerrors = []\nfor mode, factory in attempts:\n    try:\n        resp = factory()\n        text = resp.choices[0].message.content if getattr(resp, 'choices', None) else None\n        text = str(text or '').strip()\n        if not text:\n            errors.append(f'{mode}:empty')\n            continue\n        result['response'] = text[:300]\n        result['mode'] = mode\n        cleaned = re.sub(r'[^A-Z]', '', text.upper())\n        if 'GRAPH' in cleaned:\n            result['ok'] = True\n            result['reason'] = 'matched_probe_image'\n        else:\n            result['reason'] = 'response_did_not_match_probe_image'\n        print(json.dumps(result, ensure_ascii=False))\n        raise SystemExit(0)\n    except Exception as e:\n        errors.append(f'{mode}:{type(e).__name__}:{e}')\n\nresult['reason'] = '; '.join(errors)[:600]\nprint(json.dumps(result, ensure_ascii=False))", str(model_name)],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            env=env,
        )
    except subprocess.TimeoutExpired:
        return {
            'model': str(model_name),
            'ok': False,
            'response': '',
            'mode': '',
            'reason': f'timeout_after_{timeout_seconds}s',
        }
    stdout = (proc.stdout or '').strip().splitlines()
    payload = stdout[-1] if stdout else ''
    if payload:
        try:
            parsed = json.loads(payload)
            parsed['model'] = str(parsed.get('model') or model_name)
            return parsed
        except Exception:
            pass
    stderr_tail = (proc.stderr or '').strip().splitlines()
    return {
        'model': str(model_name),
        'ok': False,
        'response': '',
        'mode': '',
        'reason': stderr_tail[-1] if stderr_tail else (payload or f'probe_failed_exit_{proc.returncode}'),
    }


def list_local_vlm_models():
    candidates = [
        TASK2_DEFAULT_LOCAL_MODEL,
        'Qwen/Qwen2.5-VL-3B-Instruct',
        'Qwen/Qwen2.5-VL-7B-Instruct',
        'Qwen/Qwen2-VL-2B-Instruct',
        'Qwen/Qwen2-VL-7B-Instruct',
    ]
    seen = set()
    out = []
    for item in candidates:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out

def _escape_attr(value):
    return (
        str(value)
        .replace('&', '&amp;')
        .replace('"', '&quot;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
    )

print('REPO_DIR =', REPO_DIR)
print('SRC_DIR  =', SRC_DIR)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Task2 pipeline mode:', TASK2_NOTEBOOK_PIPELINE_MODE)
print('Task2 pipeline source:', TASK2_NOTEBOOK_PIPELINE_SOURCE)

try:
    import g4f as _g4f  # noqa: F401
    print('g4f: installed')
except Exception as _e:
    print('g4f: unavailable ->', _e)

print('Default g4f model:', TASK2_DEFAULT_G4F_MODEL)
print('Default local VLM model:', TASK2_DEFAULT_LOCAL_MODEL)
print('Default OCR backend:', getattr(_repo_settings, 'ocr_backend', 'auto'))
print('Default VLM backend:', getattr(_repo_settings, 'vlm_backend', 'g4f'))

gc.collect()


> git clone --depth 1 https://github.com/top-papers/top-papers-graph.git /content/top-papers-graph
Task2 temporal compatibility patch: not needed
> /usr/bin/python3 -m pip install -q --upgrade ipywidgets pyyaml requests panel hvplot holoviews bokeh pyvis jupyter_bokeh
> /usr/bin/python3 -m pip install -q --upgrade -e /content/top-papers-graph[task2_notebook]
> /usr/bin/python3 -m pip install -q --upgrade g4f>=7.1.2
Local Transformers/VLM stack: installed
PaddleOCR doc-parser stack missing -> paddle, paddleocr, langchain_text_splitters
> /usr/bin/python3 -m pip install -q --upgrade -i https://www.paddlepaddle.org.cn/packages/stable/cpu/ paddlepaddle==3.2.0
> /usr/bin/python3 -m pip install -q --upgrade paddleocr[doc-parser]>=3.0.0 langchain-community>=0.3 langchain-text-splitters>=0.3
> /usr/bin/python3 -m pip install -q --upgrade --force-reinstall --no-cache-dir numpy>=1.26.4,<2 pandas>=2.2.2,<2.4


REPO_DIR = /content/top-papers-graph
SRC_DIR  = /content/top-papers-graph/src
NumPy: 1.26.4
Pandas: 2.3.3
Task2 pipeline mode: full
Task2 pipeline source: /content/top-papers-graph/src/scireason/task2_validation.py
g4f: installed
Default g4f model: r1-1776
Default local VLM model: Qwen/Qwen2.5-VL-3B-Instruct
Default OCR backend: auto
Default VLM backend: g4f


86

In [2]:

# @title
# Вспомогательные функции (не нужно редактировать)
from pathlib import Path
from datetime import datetime
import ast
import json
import zipfile
import yaml

try:
    from scireason.temporal.schemas import normalize_granularity
except Exception:
    def _infer_granularity_from_value(value):
        text = str(value or '').strip()
        if not text:
            return 'year'
        if text in {'unknown', '+inf', '-inf'}:
            return 'year'
        if len(text) >= 10 and text[4:5] == '-' and text[7:8] == '-':
            return 'day'
        if len(text) >= 7 and text[4:5] == '-':
            return 'month'
        return 'year'

    def normalize_granularity(value, *, start=None, end=None, default='year'):
        text = str(value or '').strip().lower()
        aliases = {
            '': default,
            'unknown': default,
            'year': 'year',
            'annual': 'year',
            'month': 'month',
            'monthly': 'month',
            'day': 'day',
            'daily': 'day',
            'interval': 'interval',
            'range': 'interval',
            'period': 'interval',
            'timespan': 'interval',
            'time_span': 'interval',
            'date_range': 'interval',
        }
        if text in aliases:
            return aliases[text]
        inferred_start = _infer_granularity_from_value(start)
        inferred_end = _infer_granularity_from_value(end)
        if start not in (None, '') and end not in (None, '') and str(start) != str(end):
            return 'interval'
        if inferred_start == inferred_end:
            return inferred_start
        if start not in (None, '') or end not in (None, ''):
            return 'interval'
        return default


def _save_uploaded_yaml(upload_value, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)

    missing_msg = (
        'YAML-файл не загружен. Нажмите «Загрузить YAML», выберите файл '
        'с расширением .yaml или .yml и запустите кнопку ещё раз.'
    )

    if not upload_value:
        raise ValueError(missing_msg)

    if isinstance(upload_value, dict):
        if not upload_value:
            raise ValueError(missing_msg)
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            raise ValueError('Загруженный YAML пустой или повреждён. Загрузите файл заново.')
        path = out_dir / (name or 'task1.yaml')
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path

    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        name = meta.get('name') or 'task1.yaml'
        content = meta.get('content')
        if content is None:
            raise ValueError('Загруженный YAML пустой или повреждён. Загрузите файл заново.')
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path

    raise ValueError(missing_msg)


def _read_uploaded_yaml_bytes(upload_value):
    if not upload_value:
        return None, None

    if isinstance(upload_value, dict) and upload_value:
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        return name or 'task1.yaml', content

    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        if isinstance(meta, dict):
            return meta.get('name') or 'task1.yaml', meta.get('content')

    return None, None


def _format_size(num_bytes):
    if num_bytes is None:
        return None
    num = float(num_bytes)
    units = ['B', 'KB', 'MB', 'GB']
    for unit in units:
        if num < 1024 or unit == units[-1]:
            if unit == 'B':
                return f'{int(num)} {unit}'
            return f'{num:.1f} {unit}'
        num /= 1024.0


def _inspect_uploaded_yaml(upload_value):
    name, content = _read_uploaded_yaml_bytes(upload_value)
    if not name or content is None:
        return {
            'state': 'missing',
            'name': None,
            'size_bytes': None,
            'size_label': None,
            'submission_id': None,
            'topic': None,
            'message': 'YAML не загружен',
        }

    try:
        raw = content if isinstance(content, (bytes, bytearray)) else bytes(content)
    except Exception:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': None,
            'size_label': None,
            'submission_id': None,
            'topic': None,
            'message': 'Не удалось прочитать файл',
        }

    if not raw:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': 0,
            'size_label': '0 B',
            'submission_id': None,
            'topic': None,
            'message': 'Файл пустой',
        }

    size_label = _format_size(len(raw))

    try:
        text = raw.decode('utf-8')
    except UnicodeDecodeError:
        try:
            text = raw.decode('utf-8-sig')
        except UnicodeDecodeError:
            text = raw.decode('latin-1')

    try:
        parsed = yaml.safe_load(text) or {}
    except Exception as e:
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': len(raw),
            'size_label': size_label,
            'submission_id': None,
            'topic': None,
            'message': f'YAML не распознан: {type(e).__name__}',
        }

    if not isinstance(parsed, dict):
        return {
            'state': 'invalid',
            'name': name,
            'size_bytes': len(raw),
            'size_label': size_label,
            'submission_id': None,
            'topic': None,
            'message': 'В YAML ожидается объект верхнего уровня',
        }

    submission_id = parsed.get('submission_id')
    topic = parsed.get('topic')
    return {
        'state': 'valid',
        'name': name,
        'size_bytes': len(raw),
        'size_label': size_label,
        'submission_id': submission_id,
        'topic': topic,
        'message': 'YAML загружен',
    }


def _display_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    lines = [
        f"- **topic**: {manifest.get('topic')}",
        f"- **bundle_dir**: `{manifest.get('bundle_dir')}`",
        f"- **gold_graph**: `{manifest.get('gold_graph')}`",
        f"- **gold_triplets_csv**: `{manifest.get('gold_triplets_csv')}`",
    ]
    if manifest.get('auto_run_dir'):
        lines += [
            f"- **auto_run_dir**: `{manifest.get('auto_run_dir')}`",
            f"- **auto_graph_json**: `{manifest.get('auto_graph_json')}`",
            f"- **auto_triplets_csv**: `{manifest.get('auto_triplets_csv')}`",
        ]
    if manifest.get('comparison_summary'):
        lines.append(f"- **comparison_summary**: `{manifest.get('comparison_summary')}`")
    if manifest.get('reference_scout'):
        lines.append(f"- **reference_scout**: `{manifest.get('reference_scout')}`")
    if manifest.get('llm_effective_provider'):
        lines.append(f"- **llm_effective_provider**: `{manifest.get('llm_effective_provider')}`")
    if manifest.get('llm_effective_model'):
        lines.append(f"- **llm_effective_model**: `{manifest.get('llm_effective_model')}`")
    if manifest.get('vlm_effective_backend'):
        lines.append(f"- **vlm_effective_backend**: `{manifest.get('vlm_effective_backend')}`")
    if manifest.get('vlm_effective_model'):
        lines.append(f"- **vlm_effective_model**: `{manifest.get('vlm_effective_model')}`")
    display(Markdown("## Артефакты Task 2\n" + "\n".join(lines)))
    return manifest


def _safe_json_load(path):
    p = Path(path)
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding='utf-8'))


def _parse_maybe_object(value):
    if isinstance(value, (dict, list)):
        return value
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return text


def _truncate_text(value, limit=220):
    text = '' if value is None else str(value)
    text = ' '.join(text.replace('\n', ' ').split())
    if len(text) <= limit:
        return text
    return text[: max(0, limit - 1)].rstrip() + '…'


def _escape_attr(text):
    return _escape_html(text).replace('"', '&quot;').replace("'", '&#39;')


def _task2_theme_style_html():
    return """
<style>
.task2-ui, .task2-ui * { box-sizing: border-box; }
.task2-ui { color: var(--jp-ui-font-color1, #111827); }
.task2-soft { color: var(--jp-ui-font-color1, #111827); background: var(--jp-layout-color1, #ffffff); border: 1px solid var(--jp-border-color2, #d0d7de); border-radius: 8px; padding: 8px 10px; }
.task2-note { color: var(--jp-ui-font-color1, #111827); background: var(--jp-layout-color2, #f8fafc); border: 1px solid var(--jp-border-color2, #d0d7de); border-radius: 8px; padding: 8px 10px; }
.task2-small { font-size: 12px; color: var(--jp-ui-font-color2, #475569); }
.task2-titleline { font-size: 13px; color: var(--jp-ui-font-color1, #111827); line-height: 1.45; }
.task2-accent { color: var(--jp-brand-color1, #0f766e); }
.task2-prewrap { white-space: pre-wrap; word-break: break-word; }
.task2-scroll { max-height: 220px; overflow: auto; }
.task2-section { font-size: 12px; font-weight: 600; margin: 0 0 4px 0; color: var(--jp-ui-font-color1, #111827); }
.task2-help { margin-bottom: 8px; }
.task2-pill { display: inline-flex; align-items: center; gap: 6px; font-size: 12px; border-radius: 999px; padding: 2px 8px; max-width: 100%; }
.task2-pill__label { overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.task2-ui code { background: var(--jp-layout-color2, #f8fafc); border: 1px solid var(--jp-border-color2, #d0d7de); border-radius: 4px; padding: 1px 4px; }
</style>
"""


def _format_evidence_full(value):
    obj = _parse_maybe_object(value)
    if obj is None:
        return ''
    if isinstance(obj, (dict, list)):
        try:
            return json.dumps(obj, ensure_ascii=False, indent=2)
        except Exception:
            return str(obj)
    return str(obj)


def _full_text_html(label, value, max_height='180px'):
    safe_label = _escape_html(label)
    safe_value = _escape_html(value or '—')
    return (
        f"<div class='task2-soft task2-ui' style='margin:6px 0'>"
        f"<div class='task2-section'>{safe_label}</div>"
        f"<div class='task2-prewrap task2-scroll' style='max-height:{max_height}'>{safe_value}</div>"
        "</div>"
    )


def _build_fulltext_panel(row, include_raw_json=True):
    blocks = [
        _full_text_html('subject', str(row.get('subject') or ''), max_height='110px'),
        _full_text_html('predicate', str(row.get('predicate') or ''), max_height='110px'),
        _full_text_html('object', str(row.get('object') or ''), max_height='160px'),
        _full_text_html('evidence_text', str(row.get('evidence_text') or ''), max_height='220px'),
    ]
    locator = str(row.get('evidence_locator') or '')
    if locator:
        blocks.append(_full_text_html('locator', locator, max_height='120px'))
    papers_text = str(row.get('papers_text') or '')
    if papers_text:
        blocks.append(_full_text_html('papers', papers_text, max_height='140px'))
    evidence_payload = str(row.get('evidence_payload_full') or '')
    if evidence_payload:
        blocks.append(_full_text_html('evidence_payload', evidence_payload, max_height='220px'))
    if include_raw_json and row.get('raw_record_json'):
        blocks.append(_full_text_html('raw_record_json', str(row.get('raw_record_json') or ''), max_height='220px'))
    inner = W.VBox([W.HTML(value=b) for b in blocks])
    acc = W.Accordion(children=[inner])
    acc.set_title(0, 'Полный текст полей / provenance')
    return acc


def _format_evidence_short(value):
    obj = _parse_maybe_object(value)
    if isinstance(obj, dict):
        parts = []
        paper_id = obj.get('paper_id') or obj.get('id')
        snippet = obj.get('snippet_or_summary') or obj.get('snippet') or obj.get('summary') or obj.get('quote')
        page = obj.get('page')
        locator = obj.get('figure_or_table') or obj.get('locator') or obj.get('caption')
        if paper_id:
            parts.append(f"paper: {paper_id}")
        if page not in (None, '', 'nan'):
            parts.append(f"page: {page}")
        if locator:
            parts.append(f"locator: {locator}")
        if snippet:
            parts.append(str(snippet))
        text = " | ".join(parts) if parts else json.dumps(obj, ensure_ascii=False)
        return _truncate_text(text, limit=220)
    if isinstance(obj, list):
        return _truncate_text("; ".join(str(x) for x in obj[:5]), limit=220)
    return _truncate_text('' if obj is None else str(obj), limit=220)


def _read_triplets_table(path):
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    if p.suffix.lower() == '.json':
        payload = _safe_json_load(p)
        if isinstance(payload, list):
            return pd.DataFrame(payload)
        if isinstance(payload, dict):
            for key in ('assertions', 'rows', 'triplets', 'edges'):
                if isinstance(payload.get(key), list):
                    return pd.DataFrame(payload[key])
            return pd.DataFrame([payload])
    return pd.read_csv(p)


def _pick_first(row, keys, default=''):
    for key in keys:
        if key in row:
            value = row.get(key)
            if value is None:
                continue
            if isinstance(value, float) and pd.isna(value):
                continue
            text = str(value)
            if text.strip() == '':
                continue
            return value
    return default


def _normalize_assertions(df, graph_kind):
    if df is None or df.empty:
        return pd.DataFrame(columns=[
            'graph_kind', 'assertion_id', 'edge_uid', 'subject', 'predicate', 'object',
            'start_date', 'end_date', 'valid_from', 'valid_to', 'time_source',
            'time_interval', 'evidence_text', 'papers_text', 'score', 'mean_confidence',
            'verdict', 'rationale', 'time_source_note', 'raw_record_json',
            'cutoff_year', 'needs_mm_review'
        ])

    rows = []
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        row_dict = row.to_dict()
        assertion_id = str(_pick_first(row_dict, ['assertion_id', 'id'], f'{graph_kind}-{idx:05d}'))
        subject = str(_pick_first(row_dict, ['subject', 'source', 'src', 'from'], ''))
        predicate = str(_pick_first(row_dict, ['predicate', 'relation', 'label', 'type'], ''))
        object_ = str(_pick_first(row_dict, ['object', 'target', 'dst', 'to'], ''))
        start_date = str(_pick_first(row_dict, ['start_date', 'start', 'begin', 'year_start'], ''))
        end_date = str(_pick_first(row_dict, ['end_date', 'end', 'finish', 'year_end'], ''))
        valid_from = str(_pick_first(row_dict, ['valid_from', 'vf'], start_date))
        valid_to = str(_pick_first(row_dict, ['valid_to', 'vt'], end_date or '+inf'))
        time_source = str(_pick_first(row_dict, ['time_source', 'temporal_source'], ''))
        time_interval = str(_pick_first(row_dict, ['time_interval'], ''))
        papers_text = _parse_maybe_object(_pick_first(row_dict, ['papers'], []))
        if isinstance(papers_text, list):
            papers_text = ', '.join(str(x) for x in papers_text)
        evidence_raw = _pick_first(row_dict, ['evidence', 'snippet', 'quote', 'summary'], '')
        evidence_text = _format_evidence_full(evidence_raw)
        evidence_short = _format_evidence_short(evidence_raw)
        evidence_obj = _parse_maybe_object(_pick_first(row_dict, ['evidence'], {}))
        locator = ''
        needs_mm_review = False
        if isinstance(evidence_obj, dict):
            locator = str(evidence_obj.get('figure_or_table') or evidence_obj.get('locator') or evidence_obj.get('caption') or '')
            needs_mm_review = bool(locator) or any(tok in evidence_text.lower() for tok in ('figure', 'fig.', 'table', 'табл', 'рис.'))
        rows.append({
            'graph_kind': graph_kind,
            'assertion_id': assertion_id,
            'edge_uid': f'{graph_kind}:{assertion_id}',
            'subject': subject,
            'predicate': predicate,
            'object': object_,
            'subject_short': _truncate_text(subject, 90),
            'predicate_short': _truncate_text(predicate, 70),
            'object_short': _truncate_text(object_, 140),
            'start_date': start_date,
            'end_date': end_date,
            'valid_from': valid_from,
            'valid_to': valid_to,
            'time_source': time_source,
            'time_interval': time_interval,
            'evidence_text': evidence_text,
            'evidence_text_short': evidence_short,
            'evidence_payload_full': _format_evidence_full(evidence_obj if isinstance(evidence_obj, (dict, list)) else evidence_raw),
            'evidence_locator': locator,
            'papers_text': papers_text or '',
            'papers_text_short': _truncate_text(papers_text or '', 180),
            'paper_ids': _extract_paper_refs(_pick_first(row_dict, ['papers', 'paper_ids', 'paper_id'], []))[0],
            'paper_titles': _extract_paper_refs(_pick_first(row_dict, ['papers', 'paper_titles', 'title'], []))[1],
            'paper_source_refs': _extract_paper_refs(_pick_first(row_dict, ['papers', 'source_refs', 'source_ref', 'url'], []))[2],
            'score': _pick_first(row_dict, ['score'], ''),
            'mean_confidence': _pick_first(row_dict, ['mean_confidence'], ''),
            'importance_score': _pick_first(row_dict, ['importance_score'], 0),
            'importance_model': str(_pick_first(row_dict, ['importance_model'], '')),
            'importance_reasons': _parse_maybe_object(_pick_first(row_dict, ['importance_reasons'], [])),
            'topic_overlap_tokens': _parse_maybe_object(_pick_first(row_dict, ['topic_overlap_tokens'], [])),
            'verdict': str(_pick_first(row_dict, ['verdict'], '')),
            'rationale': str(_pick_first(row_dict, ['rationale'], '')),
            'time_source_note': str(_pick_first(row_dict, ['time_source_note'], '')),
            'raw_record_json': json.dumps(row_dict, ensure_ascii=False, indent=2),
            'cutoff_year': str(_pick_first(row_dict, ['cutoff_year'], '')),
            'needs_mm_review': needs_mm_review,
        })
    return pd.DataFrame(rows)


def _build_assertions_panel(assertions_df, title):
    search = W.Text(
        value='',
        description='Поиск',
        placeholder='subject / predicate / object / evidence',
        layout=W.Layout(width='52%'),
        continuous_update=False,
    )
    rows_slider = W.IntSlider(value=20, min=5, max=100, step=5, description='Строк', continuous_update=False)
    show_full = W.Checkbox(value=False, description='Показывать полный текст')
    show_details = W.Checkbox(value=True, description='Панель полного текста')
    output = W.Output()
    render_state = {'details_widget': None}

    def render(*_):
        with output:
            if render_state['details_widget'] is not None:
                _safe_close_widget_tree(render_state['details_widget'])
                render_state['details_widget'] = None
            clear_output()
            display(W.HTML(value=_task2_theme_style_html()))
            display(Markdown(f"### {title} — assertions"))
            df = assertions_df.copy()
            query = search.value.strip().lower()
            if query:
                mask = (
                    df['subject'].astype(str).str.lower().str.contains(query, na=False) |
                    df['predicate'].astype(str).str.lower().str.contains(query, na=False) |
                    df['object'].astype(str).str.lower().str.contains(query, na=False) |
                    df['evidence_text'].astype(str).str.lower().str.contains(query, na=False)
                )
                df = df[mask]
            display(W.HTML(value=f"<div class='task2-note task2-ui task2-help'><div class='task2-small'>Найдено строк: <b>{len(df)}</b>. Если в таблице текст сокращён, включите «Показывать полный текст» или раскройте панель ниже.</div></div>"))
            view = df.head(rows_slider.value).copy()
            if not show_full.value:
                for src, dst in [('subject_short', 'subject'), ('predicate_short', 'predicate'), ('object_short', 'object'), ('evidence_text_short', 'evidence_text'), ('papers_text_short', 'papers_text')]:
                    if src in view.columns:
                        view[dst] = view[src]
            show_cols = [
                'assertion_id', 'subject', 'predicate', 'object',
                'start_date', 'end_date', 'valid_from', 'valid_to',
                'time_source', 'score', 'mean_confidence', 'evidence_text'
            ]
            existing = [c for c in show_cols if c in view.columns]
            display(view[existing])
            if show_details.value and not df.empty:
                visible = df.head(rows_slider.value)
                detail_boxes = [_build_fulltext_panel(row, include_raw_json=False).children[0] for _, row in visible.iterrows()]
                accordion = W.Accordion(children=detail_boxes)
                render_state['details_widget'] = accordion
                for i, (_, row) in enumerate(visible.iterrows()):
                    label = f"{row['assertion_id']} · {row.get('subject_short') or _truncate_text(row['subject'], 60)} — {row.get('predicate_short') or _truncate_text(row['predicate'], 40)} → {row.get('object_short') or _truncate_text(row['object'], 60)}"
                    accordion.set_title(i, label[:150])
                display(W.HTML(value="<div class='task2-note task2-ui task2-help'><div class='task2-small'>Ниже можно посмотреть полный текст полей для строк из текущей таблицы.</div></div>"))
                display(accordion)

    search.observe(render, names='value')
    rows_slider.observe(render, names='value')
    show_full.observe(render, names='value')
    show_details.observe(render, names='value')
    render()
    return W.VBox([W.HTML(value=_task2_theme_style_html()), W.HBox([search, rows_slider, show_full, show_details]), output])


def _build_visualization_panel(graph_json_path, html_path, title, assertions_df=None):
    output = W.Output()
    refresh_btn = W.Button(description='Обновить визуализацию', button_style='info')
    inspector_query = W.Text(value='', description='Инспектор', placeholder='поиск по subject / object / evidence', layout=W.Layout(width='58%'), continuous_update=False)
    inspector_pick = W.Dropdown(description='Строка', options=[('—', '')], layout=W.Layout(width='42%'))
    details_output = W.Output()

    def update_inspector_options(*_):
        if assertions_df is None or assertions_df.empty:
            inspector_pick.options = [('—', '')]
            return
        df = assertions_df.copy()
        query = inspector_query.value.strip().lower()
        if query:
            mask = (
                df['subject'].astype(str).str.lower().str.contains(query, na=False) |
                df['predicate'].astype(str).str.lower().str.contains(query, na=False) |
                df['object'].astype(str).str.lower().str.contains(query, na=False) |
                df['evidence_text'].astype(str).str.lower().str.contains(query, na=False)
            )
            df = df[mask]
        options = [('—', '')]
        for _, row in df.head(20).iterrows():
            label = f"{row['assertion_id']} · {row.get('subject_short') or _truncate_text(row['subject'], 40)} — {row.get('predicate_short') or _truncate_text(row['predicate'], 25)} → {row.get('object_short') or _truncate_text(row['object'], 45)}"
            options.append((label[:160], row['edge_uid']))
        inspector_pick.options = options
        current = {v for _, v in options}
        if inspector_pick.value not in current:
            inspector_pick.value = ''

    def render_details(*_):
        with details_output:
            clear_output()
            display(W.HTML(value=_task2_theme_style_html()))
            if assertions_df is None or assertions_df.empty:
                return
            edge_uid = inspector_pick.value
            if not edge_uid:
                display(W.HTML(value="<div class='task2-note task2-ui'><div class='task2-small'>Если в графе подписи сокращены, выберите строку в инспекторе, чтобы увидеть полный текст полей.</div></div>"))
                return
            row = assertions_df[assertions_df['edge_uid'] == edge_uid].head(1)
            if row.empty:
                return
            display(_build_fulltext_panel(row.iloc[0], include_raw_json=False))

    def render_graph(*_):
        with output:
            clear_output()
            display(W.HTML(value=_task2_theme_style_html()))
            display(Markdown(f"### {title} — визуализация"))
            display(W.HTML(value="<div class='task2-note task2-ui task2-help'><div class='task2-small'>Если подписи узлов или связи в графе сокращены, используйте инспектор ниже: он показывает полный текст subject / predicate / object / evidence.</div></div>"))
            gj = Path(graph_json_path)
            hp = Path(html_path)
            if gj.exists():
                try:
                    payload = json.loads(gj.read_text(encoding='utf-8'))
                    try:
                        _, plot = make_hvplot_payload(payload)
                        if plot is not None:
                            display(plot)
                    except Exception as e:
                        display(Markdown(f'hvPlot не удалось построить: `{e}`'))
                    display(Markdown(
                        f"JSON: `{gj}`  \n"
                        f"Nodes: **{len(payload.get('nodes', []) or [])}**, edges: **{len(payload.get('edges', []) or [])}**"
                    ))
                except Exception as e:
                    display(Markdown(f'Не удалось открыть JSON графа: `{e}`'))
            else:
                display(Markdown('JSON графа не найден.'))
            if hp.exists():
                try:
                    html = hp.read_text(encoding='utf-8')
                    display(HTML(html))
                except Exception:
                    display(Markdown(f'HTML-файл графа: `{hp}`'))
            else:
                display(Markdown('HTML-визуализация не найдена.'))

    refresh_btn.on_click(render_graph)
    inspector_query.observe(update_inspector_options, names='value')
    inspector_pick.observe(render_details, names='value')
    update_inspector_options()
    render_graph()
    render_details()
    return W.VBox([W.HTML(value=_task2_theme_style_html()), refresh_btn, W.HBox([inspector_query, inspector_pick]), details_output, output])


def _build_graph_view(manifest):
    gold_df = _normalize_assertions(_read_triplets_table(manifest['gold_triplets_csv']), 'gold')
    gold_tabs = W.Tab(children=[
        _build_assertions_panel(gold_df, 'Эталонный граф'),
        _build_visualization_panel(manifest['gold_graph'], manifest['gold_graph_html'], 'Эталонный граф', gold_df),
    ])
    gold_tabs.set_title(0, 'Assertions')
    gold_tabs.set_title(1, 'Визуализация')

    graph_tabs_children = [gold_tabs]
    graph_titles = ['Эталонный граф']

    if manifest.get('auto_triplets_csv'):
        auto_df = _normalize_assertions(_read_triplets_table(manifest['auto_triplets_csv']), 'auto')
        auto_tabs = W.Tab(children=[
            _build_assertions_panel(auto_df, 'Авто-граф'),
            _build_visualization_panel(manifest['auto_graph_json'], manifest['auto_graph_html'], 'Авто-граф', auto_df),
        ])
        auto_tabs.set_title(0, 'Assertions')
        auto_tabs.set_title(1, 'Визуализация')
        graph_tabs_children.append(auto_tabs)
        graph_titles.append('Авто-граф')

    graph_tabs = W.Tab(children=graph_tabs_children)
    for idx, title in enumerate(graph_titles):
        graph_tabs.set_title(idx, title)
    return graph_tabs


def _show_reference_scout_panel(path):
    p = Path(path)
    output = W.Output()
    with output:
        if not p.exists():
            display(Markdown('Reference scout не найден.'))
        else:
            scout = json.loads(p.read_text(encoding='utf-8'))
            rows = []
            if isinstance(scout, dict):
                for hit in scout.get('hits', []):
                    for res in hit.get('results', []):
                        rows.append({
                            'query': hit.get('query'),
                            'id': res.get('id'),
                            'title': res.get('title'),
                            'year': res.get('year'),
                            'url': res.get('url'),
                        })
            elif isinstance(scout, list):
                for res in scout:
                    if not isinstance(res, dict):
                        continue
                    rows.append({
                        'query': ', '.join(res.get('trigger_queries') or []),
                        'id': res.get('paper_id') or res.get('id'),
                        'title': res.get('title'),
                        'year': res.get('year'),
                        'url': res.get('url'),
                        'score': res.get('score'),
                    })
            display(Markdown('### Reference scout'))
            if rows:
                display(pd.DataFrame(rows).head(100))
            else:
                display(Markdown('Кандидаты не найдены.'))
    return output


def _build_summary_panel(manifest):
    output = W.Output()
    with output:
        if manifest.get('comparison_summary'):
            cmp = json.loads(Path(manifest['comparison_summary']).read_text(encoding='utf-8'))
            display(Markdown('### Comparison summary'))
            display(pd.DataFrame([cmp]))
        else:
            display(Markdown('Comparison summary не найден.'))
    return output



def _step_context_blocks(task1_doc):
    blocks = []
    for idx, step in enumerate(task1_doc.get('steps', []) or [], start=1):
        if not isinstance(step, dict):
            continue
        conditions = step.get('conditions') if isinstance(step.get('conditions'), dict) else {}
        blocks.append({
            'step_id': idx,
            'claim': str(step.get('claim') or ''),
            'inference': str(step.get('inference') or ''),
            'next_question': str(step.get('next_question') or ''),
            'conditions': conditions,
        })
    return blocks


def _best_step_context(row, task1_doc):
    tokens = ' '.join([
        str(row.get('subject') or ''),
        str(row.get('predicate') or ''),
        str(row.get('object') or ''),
        str(row.get('evidence_text') or ''),
    ]).lower()
    best = None
    best_score = 0
    for block in _step_context_blocks(task1_doc):
        score = 0
        for probe in [block['claim'], block['inference'], block['next_question']]:
            for token in [t for t in str(probe).lower().replace(',', ' ').split() if len(t) > 4]:
                if token in tokens:
                    score += 1
        if score > best_score:
            best_score = score
            best = block
    return best if best_score > 0 else None


def _normalize_time_granularity(value, start_date=None, end_date=None):
    raw = str(value or "").strip().lower()
    if raw == "unknown":
        return "unknown"
    return normalize_granularity(raw, start=start_date, end=end_date, default="unknown")


def _infer_hypothesis_role(row):
    predicate = str(row.get('predicate') or '').lower()
    subj = str(row.get('subject') or '').lower()
    obj = str(row.get('object') or '').lower()
    evidence = str(row.get('evidence_text') or '').lower()
    text = ' '.join([predicate, subj, obj, evidence])
    if any(tok in text for tok in ('contrad', 'inconsistent', 'opposite', 'negative result', 'null')):
        return 'contradiction'
    if any(tok in text for tok in ('increase', 'decrease', 'effect', 'improve', 'reduce', 'cause', 'lead', 'induces')):
        return 'mechanism'
    if any(tok in text for tok in ('treatment', 'intervention', 'drug', 'stimulation', 'protocol')):
        return 'intervention'
    if any(tok in text for tok in ('assay', 'measure', 'detect', 'marker', 'readout', 'metric')):
        return 'measurement'
    if any(tok in text for tok in ('only when', 'under', 'condition', 'environment', 'temperature', 'ph', 'context')):
        return 'boundary_condition'
    return 'background'


def _infer_causal_status(row):
    predicate = str(row.get('predicate') or '').lower()
    text = ' '.join([predicate, str(row.get('evidence_text') or '').lower()])
    if any(tok in text for tok in ('cause', 'lead', 'drives', 'induces', 'mediates')):
        return 'causal'
    if any(tok in text for tok in ('associate', 'correl', 'linked', 'related')):
        return 'correlational'
    if any(tok in text for tok in ('model', 'hypothesis', 'suggest', 'propose')):
        return 'theoretical'
    return 'descriptive'


def _default_time_type(row):
    ts = str(row.get('time_source') or '').lower()
    if ts in {'metadata', 'publication_time'}:
        return 'publication_time'
    return 'observation_period'


def _html_conditions(ctx):
    if not ctx:
        return "<div class='task2-note task2-ui'><div class='task2-small'><i>Контекст шага Task 1 автоматически не сопоставлен.</i></div></div>"
    cond = ctx.get('conditions') or {}
    rows = []
    for key in ('system', 'environment', 'protocol', 'notes'):
        value = cond.get(key)
        if value:
            rows.append(f"<li><b>{_escape_html(key)}</b>: {_escape_html(str(value))}</li>")
    conditions_html = '<ul>' + ''.join(rows) + '</ul>' if rows else '<i>Условия в шаге не заданы.</i>'
    return (
        f"<div class='task2-note task2-ui'><div class='task2-small'><b>Похожий шаг Task 1:</b> step {ctx.get('step_id')}<br>"
        f"<b>claim:</b> {_escape_html(ctx.get('claim') or '')}<br>"
        f"<b>inference:</b> {_escape_html(ctx.get('inference') or '')}<br>"
        f"<b>next_question:</b> {_escape_html(ctx.get('next_question') or '')}<br>"
        f"<b>conditions:</b>{conditions_html}</div></div>"
    )

def _extract_paper_refs(value):
    obj = _parse_maybe_object(value)
    paper_ids, titles, source_refs = [], [], []

    def _push_unique(target, candidate):
        if candidate is None:
            return
        text = str(candidate).strip()
        if text and text not in target:
            target.append(text)

    def _walk(node):
        if node is None:
            return
        if isinstance(node, dict):
            _push_unique(paper_ids, node.get('paper_id') or node.get('id') or node.get('corpus_id') or node.get('doi'))
            _push_unique(titles, node.get('title') or node.get('paper_title') or node.get('name'))
            _push_unique(source_refs, node.get('source_ref') or node.get('url') or node.get('landing_page') or node.get('pdf_url'))
            for value in node.values():
                if isinstance(value, (dict, list, tuple)):
                    _walk(value)
            return
        if isinstance(node, (list, tuple)):
            for item in node:
                _walk(item)
            return
        _push_unique(paper_ids, node)

    _walk(obj)
    return paper_ids, titles, source_refs


def _default_review_state(row):
    return {
        'verdict': str(row.get('verdict') or ''),
        'rationale': str(row.get('rationale') or ''),
        'time_source_note': str(row.get('time_source_note') or ''),
        'semantic_correctness': str(row.get('semantic_correctness') or ''),
        'evidence_sufficiency': str(row.get('evidence_sufficiency') or ''),
        'scope_match': str(row.get('scope_match') or ''),
        'system_match': str(row.get('system_match') or ''),
        'environment_match': str(row.get('environment_match') or ''),
        'protocol_match': str(row.get('protocol_match') or ''),
        'scope_overgeneralized': bool(row.get('scope_overgeneralized') or False),
        'corrected_scope_note': str(row.get('corrected_scope_note') or ''),
        'hypothesis_role': str(row.get('hypothesis_role') or _infer_hypothesis_role(row)),
        'hypothesis_relevance': str(row.get('hypothesis_relevance') or '1'),
        'testability_signal': str(row.get('testability_signal') or '1'),
        'causal_status': str(row.get('causal_status') or _infer_causal_status(row)),
        'severity': str(row.get('severity') or 'warning'),
        'evidence_before_cutoff': str(row.get('evidence_before_cutoff') or ''),
        'leakage_risk': str(row.get('leakage_risk') or 'possible'),
        'time_type': str(row.get('time_type') or _default_time_type(row)),
        'time_granularity': _normalize_time_granularity(row.get('time_granularity'), start_date=row.get('start_date'), end_date=row.get('end_date')),
        'time_confidence': str(row.get('time_confidence') or 'medium'),
        'mm_verdict': str(row.get('mm_verdict') or ('needs_fix' if bool(row.get('needs_mm_review')) else '')),
        'mm_rationale': str(row.get('mm_rationale') or ''),
        'corrected_start_date': '',
        'corrected_end_date': '',
        'corrected_valid_from': '',
        'corrected_valid_to': '',
        'corrected_time_source': '',
        'correction_comment': '',
    }


def _normalize_text_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    text = str(value).strip()
    if not text:
        return []
    chunks = re.split(r'[\n,;]+', text)
    return [item.strip() for item in chunks if item.strip()]


def _parse_exclusion_spec_text(text):
    raw = str(text or '').strip()
    if not raw:
        return {}
    try:
        payload = json.loads(raw)
        return payload if isinstance(payload, dict) else {}
    except Exception:
        pass
    try:
        payload = yaml.safe_load(raw)
        return payload if isinstance(payload, dict) else {}
    except Exception:
        return {}


def _row_matches_exclusion(row, spec):
    if not spec:
        return False
    raw_json = str(row.get('raw_record_json') or '').lower()
    hay = ' '.join([
        str(row.get('subject') or ''),
        str(row.get('predicate') or ''),
        str(row.get('object') or ''),
        str(row.get('papers_text') or ''),
        str(row.get('evidence_text') or ''),
        raw_json,
    ]).lower()
    paper_ids = [str(item).lower() for item in _normalize_text_list(row.get('paper_ids'))]
    paper_titles = [str(item).lower() for item in _normalize_text_list(row.get('paper_titles'))]
    source_refs = [str(item).lower() for item in _normalize_text_list(row.get('paper_source_refs'))]

    def _contains_any(needles, values):
        norm_needles = [str(item).lower() for item in _normalize_text_list(needles)]
        return any(any(needle == value or needle in value for value in values) for needle in norm_needles)

    if _contains_any(spec.get('paper_ids') or spec.get('ids') or spec.get('articles'), paper_ids + [str(row.get('papers_text') or '').lower()]):
        return True
    if _contains_any(spec.get('titles'), paper_titles) or any(str(item).lower() in hay for item in _normalize_text_list(spec.get('titles'))):
        return True
    if _contains_any(spec.get('source_refs') or spec.get('urls'), source_refs) or any(str(item).lower() in raw_json for item in _normalize_text_list(spec.get('source_refs') or spec.get('urls'))):
        return True
    if any(str(item).lower() in hay for item in _normalize_text_list(spec.get('match_substrings') or spec.get('substrings'))):
        return True
    if any(str(item).lower() in raw_json for item in _normalize_text_list(spec.get('url_substrings'))):
        return True
    max_year = spec.get('max_year')
    if max_year not in (None, ''):
        try:
            year = int(str(row.get('start_date') or row.get('valid_from') or '')[:4])
            if year > int(max_year):
                return True
        except Exception:
            pass
    return False


def _build_validation_workspace(manifest, task1_doc):
    frames = []
    gold_df = _normalize_assertions(_read_triplets_table(manifest['gold_triplets_csv']), 'gold')
    if not gold_df.empty:
        frames.append(gold_df)
    if manifest.get('auto_triplets_csv'):
        auto_df = _normalize_assertions(_read_triplets_table(manifest['auto_triplets_csv']), 'auto')
        if not auto_df.empty:
            frames.append(auto_df)

    if not frames:
        return W.HTML(value=_task2_theme_style_html() + "<div class='task2-note task2-ui'><b>Нет данных для валидации.</b></div>")

    combined = pd.concat(frames, ignore_index=True)
    review_state = {row['edge_uid']: _default_review_state(row) for _, row in combined.iterrows()}

    reviewer_default = ''
    expert = task1_doc.get('expert') if isinstance(task1_doc.get('expert'), dict) else {}
    if expert:
        reviewer_default = str(expert.get('latin_slug') or expert.get('full_name') or '')
    reviewer_id = W.Text(value=reviewer_default, description='Reviewer', layout=W.Layout(width='55%'))
    graph_filter = W.Dropdown(
        options=[('Все графы', 'all'), ('Эталонный', 'gold'), ('Авто', 'auto')],
        value='all',
        description='Граф'
    )
    cutoff_year = str(task1_doc.get('cutoff_year') or '')
    verdict_filter = W.Dropdown(
        options=[
            ('Все', 'all'),
            ('Без решения', 'pending'),
            ('accepted', 'accepted'),
            ('rejected', 'rejected'),
            ('uncertain', 'uncertain'),
            ('needs_time_fix', 'needs_time_fix'),
            ('needs_evidence_fix', 'needs_evidence_fix'),
            ('added', 'added'),
        ],
        value='pending',
        description='Фильтр'
    )
    search = W.Text(
        value='',
        description='Поиск',
        placeholder='по subject / predicate / object / evidence',
        layout=W.Layout(width='60%'),
        continuous_update=False,
    )
    importance_threshold = W.FloatSlider(value=float((manifest.get('filter_defaults') or {}).get('importance_threshold') or 0.0), min=0.0, max=1.0, step=0.05, description='Важн. ≥', readout_format='.2f', continuous_update=False)
    exclusion_text = W.Textarea(
        value=json.dumps((manifest.get('filter_defaults') or {}).get('exclusion_rules') or {}, ensure_ascii=False, indent=2) if (manifest.get('filter_defaults') or {}).get('exclusion_rules') else '',
        description='Исключения',
        placeholder='paper_ids\n  - PMID:12345\nmatch_substrings\n  - future review',
        layout=W.Layout(width='100%', height='110px'),
        continuous_update=False,
    )
    exclusion_upload = W.FileUpload(accept='.yaml,.yml,.json', multiple=False, description='YAML исключений')
    exclusion_hint = W.HTML(value='<div class="task2-note task2-ui"><div class="task2-small">Исключите статьи и рёбра из будущего с помощью YAML/JSON: <code>paper_ids</code>, <code>titles</code>, <code>source_refs</code>, <code>match_substrings</code>, <code>url_substrings</code>.</div></div>')
    page_size = W.IntSlider(value=5, min=1, max=10, step=1, description='На стр.', continuous_update=False)
    page = W.IntSlider(value=1, min=1, max=1, step=1, description='Стр.', continuous_update=False)
    summary_html = W.HTML()
    editor_output = W.Output(layout=W.Layout(border='1px solid var(--jp-border-color2, #e2e8f0)', padding='8px'))
    export_status = W.HTML()
    export_btn = W.Button(description='Сохранить файлы валидации', button_style='success')
    refresh_btn = W.Button(description='Обновить список', button_style='info')
    save_draft_btn = W.Button(description='Сохранить черновик', button_style='')
    load_draft_btn = W.Button(description='Загрузить черновик', button_style='')
    draft_path = W.Text(
        value=str(manifest.get('review_state_latest') or get_task2_review_state_paths(manifest['bundle_dir'])['latest']),
        description='Draft JSON',
        layout=W.Layout(width='100%')
    )
    autosave_toggle = W.Checkbox(value=True, description='Автосохранение черновика')
    download_btn = W.Button(description='Скачать ZIP', button_style='warning')
    download_btn.disabled = True
    offline_form_btn = W.Button(description='Скачать автономную форму', button_style='primary')
    render_cache = {'widget_roots': []}

    def _ensure_offline_form_path():
        try:
            current = manifest.get('offline_review_html')
            if current and Path(current).exists():
                return Path(current)
            generated = build_task2_offline_review_package(manifest, task1_doc)
            manifest['offline_review_html'] = str(generated)
            return Path(generated)
        except Exception:
            return None

    offline_form_btn.disabled = _ensure_offline_form_path() is None

    def _review_state_payload(reason='manual'):
        return {
            'artifact_version': 1,
            'reason': reason,
            'reviewer_id': reviewer_id.value.strip(),
            'filters': {
                'graph_filter': graph_filter.value,
                'verdict_filter': verdict_filter.value,
                'search': search.value,
                'importance_threshold': float(importance_threshold.value),
                'exclusion_text': exclusion_text.value,
                'page_size': int(page_size.value),
                'page': int(page.value),
            },
            'review_state': review_state,
        }

    def _persist_review_state(reason='manual', force=False):
        if not force and not autosave_toggle.value:
            return None
        target = Path(draft_path.value).expanduser()
        target.parent.mkdir(parents=True, exist_ok=True)
        saved = save_task2_review_state(manifest['bundle_dir'], _review_state_payload(reason=reason), label=reason)
        payload = load_task2_review_state(manifest['bundle_dir'])
        encoded = json.dumps(payload, ensure_ascii=False, indent=2)
        target.write_text(encoded, encoding='utf-8')
        draft_path.value = str(target)
        export_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">'
            f'<b>Черновик сохранён.</b> <code>{_escape_html(str(target))}</code>'
            '</div></div>'
        )
        return saved

    def _apply_loaded_state(payload):
        loaded_state = payload.get('review_state') or {}
        for edge_uid, state in loaded_state.items():
            if edge_uid in review_state and isinstance(state, dict):
                merged = dict(_default_review_state(combined[combined['edge_uid'] == edge_uid].iloc[0]))
                merged.update(state)
                review_state[edge_uid] = merged
        reviewer_id.value = str(payload.get('reviewer_id') or reviewer_id.value or '')
        filters = payload.get('filters') or {}
        if filters.get('graph_filter') in {'all', 'gold', 'auto'}:
            graph_filter.value = filters.get('graph_filter')
        if filters.get('verdict_filter') in {'all', 'pending', 'accepted', 'rejected', 'uncertain', 'needs_time_fix', 'needs_evidence_fix', 'added'}:
            verdict_filter.value = filters.get('verdict_filter')
        search.value = str(filters.get('search') or '')
        try:
            importance_threshold.value = float(filters.get('importance_threshold') or importance_threshold.value)
        except Exception:
            pass
        exclusion_text.value = str(filters.get('exclusion_text') or exclusion_text.value or '')
        try:
            page_size.value = int(filters.get('page_size') or page_size.value)
        except Exception:
            pass
        update_summary()
        render_editors()
        try:
            page.value = max(1, int(filters.get('page') or page.value))
        except Exception:
            pass

    def _load_review_state_from_path(_=None):
        target = Path(draft_path.value).expanduser()
        if not target.exists():
            export_status.value = (
                '<div class="task2-note task2-ui"><div class="task2-small">'
                f'Черновик не найден: <code>{_escape_html(str(target))}</code>'
                '</div></div>'
            )
            return
        payload = json.loads(target.read_text(encoding='utf-8'))
        _apply_loaded_state(payload)
        export_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">'
            f'<b>Черновик загружен.</b> <code>{_escape_html(str(target))}</code>'
            '</div></div>'
        )

    def filtered_df():
        df = combined.copy()
        if graph_filter.value != 'all':
            df = df[df['graph_kind'] == graph_filter.value]

        threshold = float(importance_threshold.value or 0.0)
        if 'importance_score' in df.columns:
            numeric_score = pd.to_numeric(df['importance_score'], errors='coerce').fillna(0.0)
            df = df[numeric_score >= threshold]

        exclusion_spec = _parse_exclusion_spec_text(exclusion_text.value)
        if exclusion_spec:
            mask = [not _row_matches_exclusion(row, exclusion_spec) for _, row in df.iterrows()]
            df = df[pd.Series(mask, index=df.index)]

        query = search.value.strip().lower()
        if query:
            mask = (
                df['subject'].astype(str).str.lower().str.contains(query, na=False) |
                df['predicate'].astype(str).str.lower().str.contains(query, na=False) |
                df['object'].astype(str).str.lower().str.contains(query, na=False) |
                df['evidence_text'].astype(str).str.lower().str.contains(query, na=False) |
                df['papers_text'].astype(str).str.lower().str.contains(query, na=False)
            )
            df = df[mask]

        if verdict_filter.value != 'all':
            if verdict_filter.value == 'pending':
                mask = [not review_state[row['edge_uid']]['verdict'] for _, row in df.iterrows()]
                df = df[pd.Series(mask, index=df.index)]
            else:
                mask = [review_state[row['edge_uid']]['verdict'] == verdict_filter.value for _, row in df.iterrows()]
                df = df[pd.Series(mask, index=df.index)]

        return df.reset_index(drop=True)

    def make_observer(edge_uid, key):
        def _observer(change):
            review_state[edge_uid][key] = change['new']
            update_summary()
            _persist_review_state(reason='autosave')
        return _observer

    def update_summary(*_):
        total = len(combined)
        decided = sum(1 for st in review_state.values() if st['verdict'])
        corrected = sum(
            1 for st in review_state.values()
            if any(st.get(k) for k in (
                'corrected_start_date', 'corrected_end_date',
                'corrected_valid_from', 'corrected_valid_to',
                'corrected_time_source', 'correction_comment'
            ))
        )
        pending = total - decided
        severity_violation = sum(1 for st in review_state.values() if st.get('severity') == 'violation')
        current_rows = len(filtered_df())
        summary_html.value = (
            '<div class="task2-ui task2-small" style="padding:6px 0">'
            f'<b>Всего рёбер:</b> {total} &nbsp; '
            f'<b>С оценкой:</b> {decided} &nbsp; '
            f'<b>Без оценки:</b> {pending} &nbsp; '
            f'<b>С временными правками:</b> {corrected} &nbsp; '
            f'<b>Severity=violation:</b> {severity_violation} &nbsp; '
            f'<b>Важность ≥</b> {float(importance_threshold.value):.2f} &nbsp; '
            f'<b>После фильтров:</b> {current_rows}'
            '</div></div>'
        )

    def render_editors(*_):
        df = filtered_df()
        total_pages = max(1, int((len(df) + page_size.value - 1) / page_size.value))
        page.max = total_pages
        if page.value > total_pages:
            page.value = total_pages
        start = (page.value - 1) * page_size.value
        visible = df.iloc[start:start + page_size.value]

        with editor_output:
            for old_widget in render_cache['widget_roots']:
                _safe_close_widget_tree(old_widget)
            render_cache['widget_roots'] = []
            clear_output()
            display(W.HTML(value=_task2_theme_style_html()))
            display(Markdown(
                '### Валидация рёбер\n'
                'Для каждого ребра выберите verdict и при необходимости внесите корректировки временных полей. '
                'После этого нажмите **«Сохранить файлы валидации»**.'
            ))
            display(W.HTML(value="<div class='task2-note task2-ui task2-help'><div class='task2-small'>Заголовки карточек могут быть сокращены для компактности. Полный текст полей доступен внутри каждой карточки в секции «Полный текст полей / provenance».</div></div>"))
            if visible.empty:
                display(Markdown('По текущим фильтрам строки не найдены.'))
                return

            items = []
            for _, row in visible.iterrows():
                state = review_state[row['edge_uid']]

                header_text = f"[{row['graph_kind']}] {row['subject']} — {row['predicate']} → {row['object']}"
                header = W.HTML(
                    value=(
                        f"<div class='task2-titleline task2-ui task2-prewrap' title='{_escape_attr(header_text)}'><b>[{row['graph_kind']}]</b> "
                        f"{_escape_html(row['subject'])} "
                        f"<span class='task2-accent'>— { _escape_html(row['predicate']) } →</span> "
                        f"{_escape_html(row['object'])}</div>"
                    )
                )
                ctx = _best_step_context(row, task1_doc)
                meta = W.HTML(
                    value=(
                        "<div class='task2-note task2-ui'><div class='task2-small'>"
                        f"assertion_id: <code>{_escape_html(row['assertion_id'])}</code> · "
                        f"start/end: <code>{_escape_html(row['start_date'])}</code> / <code>{_escape_html(row['end_date'])}</code> · "
                        f"valid: <code>{_escape_html(row['valid_from'])}</code> / <code>{_escape_html(row['valid_to'])}</code> · "
                        f"time_source: <code>{_escape_html(row['time_source'])}</code> · "
                        f"importance: <code>{_escape_html('{:.3f}'.format(float(row.get('importance_score') or 0)))}</code>"
                        "</div></div>"
                    )
                )
                context_html = W.HTML(value=_html_conditions(ctx))
                evidence = W.HTML(
                    value=f"<div class='task2-note task2-ui'><div class='task2-small'><b>Evidence (коротко):</b> {_escape_html(_truncate_text(row['evidence_text'] or '—', 320))}</div></div>"
                )

                verdict = W.Dropdown(
                    options=[
                        ('', ''),
                        ('accepted', 'accepted'),
                        ('rejected', 'rejected'),
                        ('uncertain', 'uncertain'),
                        ('needs_time_fix', 'needs_time_fix'),
                        ('needs_evidence_fix', 'needs_evidence_fix'),
                        ('added', 'added'),
                    ],
                    value=state['verdict'],
                    description='Verdict',
                    layout=W.Layout(width='280px')
                )
                semantic = W.Dropdown(
                    options=[('', ''), ('correct', 'correct'), ('incorrect', 'incorrect'), ('unclear', 'unclear')],
                    value=state['semantic_correctness'], description='Смысл', layout=W.Layout(width='220px')
                )
                evidence_suff = W.Dropdown(
                    options=[('', ''), ('sufficient', 'sufficient'), ('weak', 'weak'), ('wrong_evidence', 'wrong_evidence'), ('needs_multimodal_evidence', 'needs_multimodal_evidence')],
                    value=state['evidence_sufficiency'], description='Evidence', layout=W.Layout(width='300px')
                )
                rationale = W.Textarea(
                    value=state['rationale'],
                    description='Почему',
                    placeholder='Коротко: что подтверждается, что не подтверждается и почему это важно для гипотезы',
                    layout=W.Layout(width='100%', height='90px'),
                    continuous_update=False,
                )
                time_source_note = W.Text(
                    value=state['time_source_note'],
                    description='Источник времени',
                    placeholder='explicit_text / figure_or_table / metadata / expert_inference / unknown',
                    continuous_update=False,
                )
                evidence_before_cutoff = W.Dropdown(
                    options=[('', ''), ('yes', 'yes'), ('no', 'no'), ('unclear', 'unclear')],
                    value=state['evidence_before_cutoff'], description='До cutoff', layout=W.Layout(width='200px')
                )
                leakage_risk = W.Dropdown(
                    options=[('none', 'none'), ('possible', 'possible'), ('high', 'high')],
                    value=state['leakage_risk'], description='Leakage', layout=W.Layout(width='220px')
                )
                time_type = W.Dropdown(
                    options=[('observation_period', 'observation_period'), ('publication_time', 'publication_time'), ('historical_fact', 'historical_fact'), ('theory_prediction', 'theory_prediction'), ('retrospective_review', 'retrospective_review')],
                    value=state['time_type'], description='Time type', layout=W.Layout(width='280px')
                )
                time_gran = W.Dropdown(
                    options=[('unknown', 'unknown'), ('year', 'year'), ('month', 'month'), ('day', 'day'), ('interval', 'interval')],
                    value=state['time_granularity'], description='Granularity', layout=W.Layout(width='230px')
                )
                time_conf = W.Dropdown(
                    options=[('high', 'high'), ('medium', 'medium'), ('low', 'low')],
                    value=state['time_confidence'], description='Time conf', layout=W.Layout(width='220px')
                )
                scope_match = W.Dropdown(
                    options=[('', ''), ('yes', 'yes'), ('partial', 'partial'), ('no', 'no')],
                    value=state['scope_match'], description='Scope', layout=W.Layout(width='180px')
                )
                system_match = W.Dropdown(
                    options=[('', ''), ('yes', 'yes'), ('partial', 'partial'), ('no', 'no')],
                    value=state['system_match'], description='System', layout=W.Layout(width='180px')
                )
                environment_match = W.Dropdown(
                    options=[('', ''), ('yes', 'yes'), ('partial', 'partial'), ('no', 'no')],
                    value=state['environment_match'], description='Env', layout=W.Layout(width='180px')
                )
                protocol_match = W.Dropdown(
                    options=[('', ''), ('yes', 'yes'), ('partial', 'partial'), ('no', 'no')],
                    value=state['protocol_match'], description='Protocol', layout=W.Layout(width='180px')
                )
                scope_over = W.Checkbox(value=state['scope_overgeneralized'], description='Есть overgeneralization')
                scope_note = W.Textarea(value=state['corrected_scope_note'], description='Scope note', layout=W.Layout(width='100%', height='70px'), continuous_update=False)
                hyp_role = W.Dropdown(
                    options=[('background', 'background'), ('mechanism', 'mechanism'), ('intervention', 'intervention'), ('measurement', 'measurement'), ('outcome', 'outcome'), ('boundary_condition', 'boundary_condition'), ('contradiction', 'contradiction'), ('null_result', 'null_result')],
                    value=state['hypothesis_role'], description='Role', layout=W.Layout(width='260px')
                )
                hyp_rel = W.Dropdown(options=[('0', '0'), ('1', '1'), ('2', '2')], value=state['hypothesis_relevance'], description='Usefulness', layout=W.Layout(width='170px'))
                testability = W.Dropdown(options=[('0', '0'), ('1', '1'), ('2', '2')], value=state['testability_signal'], description='Testability', layout=W.Layout(width='170px'))
                causal = W.Dropdown(options=[('causal', 'causal'), ('correlational', 'correlational'), ('theoretical', 'theoretical'), ('descriptive', 'descriptive'), ('unclear', 'unclear')], value=state['causal_status'], description='Causality', layout=W.Layout(width='250px'))
                severity = W.Dropdown(options=[('info', 'info'), ('warning', 'warning'), ('violation', 'violation')], value=state['severity'], description='Severity', layout=W.Layout(width='200px'))
                mm_verdict = W.Dropdown(options=[('', ''), ('accepted', 'accepted'), ('needs_fix', 'needs_fix'), ('rejected', 'rejected')], value=state['mm_verdict'], description='MM review', layout=W.Layout(width='230px'))
                mm_rationale = W.Textarea(value=state['mm_rationale'], description='MM note', layout=W.Layout(width='100%', height='60px'), continuous_update=False)

                corrected_start = W.Text(value=state['corrected_start_date'], description='start_date')
                corrected_end = W.Text(value=state['corrected_end_date'], description='end_date')
                corrected_vf = W.Text(value=state['corrected_valid_from'], description='valid_from')
                corrected_vt = W.Text(value=state['corrected_valid_to'], description='valid_to')
                corrected_ts = W.Text(value=state['corrected_time_source'], description='time_source')
                correction_comment = W.Textarea(
                    value=state['correction_comment'],
                    description='Правка',
                    placeholder='Что именно нужно исправить во времени / provenance / условиях',
                    layout=W.Layout(width='100%', height='70px'),
                    continuous_update=False,
                )

                widgets_for_state = [
                    (verdict, 'verdict'), (semantic, 'semantic_correctness'), (evidence_suff, 'evidence_sufficiency'),
                    (rationale, 'rationale'), (time_source_note, 'time_source_note'), (evidence_before_cutoff, 'evidence_before_cutoff'),
                    (leakage_risk, 'leakage_risk'), (time_type, 'time_type'), (time_gran, 'time_granularity'), (time_conf, 'time_confidence'),
                    (scope_match, 'scope_match'), (system_match, 'system_match'), (environment_match, 'environment_match'),
                    (protocol_match, 'protocol_match'), (scope_over, 'scope_overgeneralized'), (scope_note, 'corrected_scope_note'),
                    (hyp_role, 'hypothesis_role'), (hyp_rel, 'hypothesis_relevance'), (testability, 'testability_signal'),
                    (causal, 'causal_status'), (severity, 'severity'), (mm_verdict, 'mm_verdict'), (mm_rationale, 'mm_rationale'),
                    (corrected_start, 'corrected_start_date'), (corrected_end, 'corrected_end_date'),
                    (corrected_vf, 'corrected_valid_from'), (corrected_vt, 'corrected_valid_to'),
                    (corrected_ts, 'corrected_time_source'), (correction_comment, 'correction_comment'),
                ]
                for widget, key in widgets_for_state:
                    widget.observe(make_observer(row['edge_uid'], key), names='value')

                reset_btn = W.Button(description='Очистить правки', button_style='')
                def _make_reset(edge_uid, widgets):
                    def _reset(_):
                        review_state[edge_uid] = _default_review_state(row)
                        defaults = review_state[edge_uid]
                        for w, key in widgets:
                            w.value = defaults[key]
                        update_summary()
                        _persist_review_state(reason='reset')
                    return _reset
                reset_btn.on_click(_make_reset(row['edge_uid'], widgets_for_state))

                box = W.VBox([
                    header,
                    meta,
                    context_html,
                    evidence,
                    _build_fulltext_panel(row),
                    W.HBox([verdict, semantic, evidence_suff, severity, reset_btn]),
                    rationale,
                    W.HTML('<b>Temporal validity and leakage</b>'),
                    W.HBox([evidence_before_cutoff, leakage_risk, time_type, time_gran, time_conf]),
                    time_source_note,
                    W.HTML('<b>Conditions / scope from Task 1</b>'),
                    W.HBox([scope_match, system_match, environment_match, protocol_match]),
                    scope_over,
                    scope_note,
                    W.HTML('<b>Hypothesis usefulness</b>'),
                    W.HBox([hyp_role, hyp_rel, testability, causal]),
                    W.HTML('<b>Multimodal evidence</b>'),
                    W.HBox([mm_verdict]),
                    mm_rationale,
                    W.HTML('<b>Коррекция временных параметров</b>'),
                    W.HBox([corrected_start, corrected_end]),
                    W.HBox([corrected_vf, corrected_vt]),
                    corrected_ts,
                    correction_comment,
                ])
                items.append(box)
                render_cache['widget_roots'].append(box)

            accordion = W.Accordion(children=items)
            render_cache['widget_roots'].append(accordion)
            for i, (_, row) in enumerate(visible.iterrows()):
                accordion.set_title(i, _truncate_text(f"[{row['graph_kind']}] {row['subject']} — {row['predicate']} → {row['object']}", 120))
            display(accordion)

    def _build_export_frames():
        review_rows = []
        correction_rows = []
        for _, row in combined.iterrows():
            state = review_state[row['edge_uid']]
            merged = row.to_dict()
            merged.update({
                'reviewer_id': reviewer_id.value.strip(),
                'review_timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
                'expert_verdict': state['verdict'],
                'expert_rationale': state['rationale'],
                'expert_time_source_note': state['time_source_note'],
                'corrected_start_date': state['corrected_start_date'],
                'corrected_end_date': state['corrected_end_date'],
                'corrected_valid_from': state['corrected_valid_from'],
                'corrected_valid_to': state['corrected_valid_to'],
                'corrected_time_source': state['corrected_time_source'],
                'correction_comment': state['correction_comment'],
            })
            review_rows.append(merged)

            has_correction = any(state.get(k) for k in [
                'corrected_start_date', 'corrected_end_date', 'corrected_valid_from',
                'corrected_valid_to', 'corrected_time_source', 'correction_comment'
            ])
            if has_correction:
                correction_rows.append({
                    'edge_uid': row['edge_uid'],
                    'graph_kind': row['graph_kind'],
                    'assertion_id': row['assertion_id'],
                    'subject': row['subject'],
                    'predicate': row['predicate'],
                    'object': row['object'],
                    'original_start_date': row['start_date'],
                    'original_end_date': row['end_date'],
                    'original_valid_from': row['valid_from'],
                    'original_valid_to': row['valid_to'],
                    'corrected_start_date': state['corrected_start_date'],
                    'corrected_end_date': state['corrected_end_date'],
                    'corrected_valid_from': state['corrected_valid_from'],
                    'corrected_valid_to': state['corrected_valid_to'],
                    'corrected_time_source': state['corrected_time_source'],
                    'comment': state['correction_comment'],
                    'rationale': state['rationale'],
                    'reviewer_id': reviewer_id.value.strip(),
                })

        return pd.DataFrame(review_rows), pd.DataFrame(correction_rows)

    last_export = {'zip_path': None}

    def export_reviews(_):
        review_df, corrections_df = _build_export_frames()
        bundle_dir = Path(manifest['bundle_dir'])
        export_dir = bundle_dir / 'expert_validation'
        export_dir.mkdir(parents=True, exist_ok=True)

        review_csv = export_dir / 'edge_reviews.csv'
        review_json = export_dir / 'edge_reviews.json'
        corrections_csv = export_dir / 'temporal_corrections.csv'
        corrections_json = export_dir / 'temporal_corrections.json'
        summary_json = export_dir / 'validation_summary.json'
        zip_path = export_dir / 'expert_validation_bundle.zip'

        _persist_review_state(reason='export', force=True)
        review_df.to_csv(review_csv, index=False)
        corrections_df.to_csv(corrections_csv, index=False)

        review_payload = {
            'artifact_version': 5,
            'domain': str(task1_doc.get('domain') or ''),
            'topic': str(task1_doc.get('topic') or ''),
            'trajectory_submission_id': str(task1_doc.get('submission_id') or ''),
            'cutoff_year': str(task1_doc.get('cutoff_year') or ''),
            'reviewer_id': reviewer_id.value.strip(),
            'timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
            'filter_settings': {
                'importance_threshold': float(importance_threshold.value),
                'exclusion_rules': _parse_exclusion_spec_text(exclusion_text.value),
            },
            'assertions': review_df.to_dict(orient='records'),
            'added_edges': review_df[review_df['expert_verdict'].fillna('') == 'added'].to_dict(orient='records') if 'expert_verdict' in review_df else [],
        }
        corrections_payload = {
            'artifact_version': 3,
            'domain': str(task1_doc.get('domain') or ''),
            'paper_id': '',
            'reviewer_id': reviewer_id.value.strip(),
            'trajectory_submission_id': str(task1_doc.get('submission_id') or ''),
            'corrections': corrections_df.to_dict(orient='records'),
        }
        summary_payload = {
            'total_edges': int(len(review_df)),
            'decided_edges': int((review_df['expert_verdict'].fillna('') != '').sum()) if 'expert_verdict' in review_df else 0,
            'accepted_edges': int((review_df['expert_verdict'].fillna('') == 'accepted').sum()) if 'expert_verdict' in review_df else 0,
            'rejected_edges': int((review_df['expert_verdict'].fillna('') == 'rejected').sum()) if 'expert_verdict' in review_df else 0,
            'uncertain_edges': int((review_df['expert_verdict'].fillna('') == 'uncertain').sum()) if 'expert_verdict' in review_df else 0,
            'needs_time_fix_edges': int((review_df['expert_verdict'].fillna('') == 'needs_time_fix').sum()) if 'expert_verdict' in review_df else 0,
            'needs_evidence_fix_edges': int((review_df['expert_verdict'].fillna('') == 'needs_evidence_fix').sum()) if 'expert_verdict' in review_df else 0,
            'added_edges': int((review_df['expert_verdict'].fillna('') == 'added').sum()) if 'expert_verdict' in review_df else 0,
            'severity_violation_edges': int((review_df['severity'].fillna('') == 'violation').sum()) if 'severity' in review_df else 0,
            'high_hypothesis_relevance_edges': int((review_df['hypothesis_relevance'].astype(str) == '2').sum()) if 'hypothesis_relevance' in review_df else 0,
            'multimodal_needs_fix_edges': int((review_df['mm_verdict'].fillna('') == 'needs_fix').sum()) if 'mm_verdict' in review_df else 0,
            'temporal_corrections': int(len(corrections_df)),
            'active_filters': {
                'importance_threshold': float(importance_threshold.value),
                'exclusion_rules': _parse_exclusion_spec_text(exclusion_text.value),
            },
            'export_dir': str(export_dir),
        }

        review_json.write_text(json.dumps(review_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        corrections_json.write_text(json.dumps(corrections_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

        with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            for p in [review_csv, review_json, corrections_csv, corrections_json, summary_json]:
                zf.write(p, arcname=p.name)

        last_export['zip_path'] = zip_path
        download_btn.disabled = False
        export_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small"><b>Файлы валидации сохранены.</b><br>'
            f'CSV с оценкой рёбер: <code>{review_csv}</code><br>'
            f'CSV с временными правками: <code>{corrections_csv}</code><br>'
            f'ZIP для выгрузки: <code>{zip_path}</code>'
            '</div>'
        )

    def download_export(_):
        zip_path = last_export.get('zip_path')
        if not zip_path:
            export_status.value = '<div class="task2-note task2-ui"><div class="task2-small">Сначала сохраните файлы валидации.</div></div>'
            return
        try:
            from google.colab import files as colab_files
            colab_files.download(str(zip_path))
        except Exception:
            export_status.value += (
                '<div class="task2-note task2-ui"><div class="task2-small">Автоскачивание недоступно в этом окружении. '
                f'Заберите ZIP по пути: <code>{zip_path}</code></div></div>'
            )

    def download_offline_form(_):
        html_path = _ensure_offline_form_path()
        if html_path is None:
            export_status.value = (
                '<div class="task2-note task2-ui"><div class="task2-small">'
                '<b>Не удалось подготовить автономную форму.</b> Проверьте, что в репозитории есть обновлённый модуль Task 2.'
                '</div></div>'
            )
            return
        try:
            from google.colab import files as colab_files
            colab_files.download(str(html_path))
        except Exception:
            export_status.value += (
                '<div class="task2-note task2-ui"><div class="task2-small">Автоскачивание HTML недоступно в этом окружении. '
                f'Заберите файл по пути: <code>{html_path}</code></div></div>'
            )

    def _update_exclusion_upload(change):
        value = change.get('new')
        if not value:
            return
        try:
            if isinstance(value, dict):
                meta = next(iter(value.values()))
            else:
                meta = value[0]
            content = meta.get('content')
            if content is not None:
                exclusion_text.value = (content.decode('utf-8') if isinstance(content, (bytes, bytearray)) else bytes(content).decode('utf-8'))
        except Exception as exc:
            export_status.value = f'<div class="task2-note task2-ui"><div class="task2-small"><b>Не удалось прочитать exclusions файл:</b> {exc}</div></div>'

    exclusion_upload.observe(_update_exclusion_upload, names='value')

    for widget in [graph_filter, verdict_filter, search, page_size, importance_threshold, exclusion_text]:
        widget.observe(render_editors, names='value')
        widget.observe(update_summary, names='value')
    page.observe(render_editors, names='value')
    refresh_btn.on_click(render_editors)
    export_btn.on_click(export_reviews)
    download_btn.on_click(download_export)
    offline_form_btn.on_click(download_offline_form)
    save_draft_btn.on_click(lambda _: _persist_review_state(reason='manual', force=True))
    load_draft_btn.on_click(_load_review_state_from_path)

    update_summary()
    render_editors()
    latest_draft = Path(draft_path.value).expanduser()
    if latest_draft.exists():
        _load_review_state_from_path()

    return W.VBox([
        W.HTML(value=_task2_theme_style_html()),
        W.HTML(
            '<div class="task2-note task2-ui" style="margin:6px 0 10px 0">'
            '<div class="task2-small">Здесь эксперт оценивает каждое ребро и при необходимости корректирует его временные параметры. '
            'Если в заголовках карточек или графах текст сокращён, полный текст доступен в секции «Полный текст полей / provenance». '
            'На выходе ноутбук сформирует готовые CSV/JSON и ZIP с результатами. '
            'Отдельно можно скачать автономную HTML-форму и заполнить её уже на устройстве эксперта.</div>'
            '</div>'
        ),
        W.HBox([reviewer_id, summary_html]),
        W.HBox([graph_filter, verdict_filter, importance_threshold]),
        W.HBox([search, page_size, page]),
        exclusion_hint,
        exclusion_text,
        W.HBox([exclusion_upload]),
        W.HBox([refresh_btn, save_draft_btn, load_draft_btn, export_btn, download_btn, offline_form_btn]),
        W.HBox([draft_path, autosave_toggle]),
        export_status,
        editor_output,
    ])


def _build_review_workspace(manifest, task1_doc):
    children = [
        _build_graph_view(manifest),
        _build_validation_workspace(manifest, task1_doc),
    ]
    titles = ['Просмотр графов', 'Валидация эксперта']

    if manifest.get('comparison_summary'):
        children.append(_build_summary_panel(manifest))
        titles.append('Сводка')
    if manifest.get('reference_scout'):
        children.append(_show_reference_scout_panel(manifest['reference_scout']))
        titles.append('Reference scout')

    tabs = W.Tab(children=children)
    for idx, title in enumerate(titles):
        tabs.set_title(idx, title)
    return tabs


In [3]:
import re

# @title
# Форма Task 2 (запустите ячейку и заполните поля)

yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
yaml_indicator = W.HTML()
exclusion_upload = W.FileUpload(accept='.yaml,.yml,.json', multiple=False, description='YAML исключений')
exclusion_hint = W.HTML('<div class="task2-note task2-ui"><div class="task2-small">Необязательно: загрузите YAML/JSON c исключениями по статьям, чтобы убрать триплеты из статей будущего и исключить leakage. Поддерживаются ключи <code>paper_ids</code>, <code>titles</code>, <code>source_refs</code>, <code>match_substrings</code>, <code>url_substrings</code>.</div></div>')
importance_threshold = W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='Важн. ≥', readout_format='.2f', continuous_update=False)
out_dir = W.Text(value='runs/task2_validation', description='Out dir', layout=W.Layout(width='100%'))
run_auto = W.Checkbox(value=True, description='Строить автоматический temporal KG')
reference_scout = W.Checkbox(value=True, description='Сгенерировать reference scout')
remote_lookup = W.Checkbox(value=True, description='Универсальный онлайн-резолв статей (DOI/URL/source-hints)')
multimodal = W.Checkbox(value=True, description='Использовать multimodal pipeline')


def _save_optional_upload(upload_value, out_dir: Path, default_name: str) -> Path | None:
    if not upload_value:
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        if not upload_value:
            return None
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            return None
        path = out_dir / (name or default_name)
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        content = meta.get('content')
        if content is None:
            return None
        name = meta.get('name') or default_name
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    raise TypeError('Неподдерживаемый формат upload value')


model_source = W.Dropdown(
    options=[
        ('g4f (default)', 'g4f'),
        ('Local / Transformers', 'local'),
    ],
    value='g4f',
    description='MM model',
)
g4f_refresh_btn = W.Button(description='Проверить рабочие g4f модели', icon='search')
local_reset_btn = W.Button(description='Сбросить local model', icon='undo')
g4f_model = W.Dropdown(
    options=_task2_build_g4f_dropdown_options(prefer_verified=False, include_auto=True),
    value=TASK2_DEFAULT_G4F_MODEL if TASK2_DEFAULT_G4F_MODEL in [value for _, value in _task2_build_g4f_dropdown_options(prefer_verified=False, include_auto=True)] else 'auto',
    description='g4f model',
    layout=W.Layout(width='100%'),
)
local_model = W.Combobox(
    options=list_local_vlm_models(),
    value=TASK2_DEFAULT_LOCAL_MODEL,
    placeholder='например, Qwen/Qwen2.5-VL-3B-Instruct',
    description='Local model',
    layout=W.Layout(width='100%'),
)
g4f_probe_status = W.HTML()
model_hint = W.HTML()
run_btn = W.Button(description='Проверить форму', button_style='info')
status = W.HTML()
progress_label = W.HTML()
progress_bar = W.IntProgress(value=0, min=0, max=100, description='Прогресс', bar_style='info', layout=W.Layout(width='100%'))
progress_log = W.HTML()
out = W.Output()
_TASK2_LAST_WORKSPACE = None


def _yaml_indicator_chip(color_text, color_bg, color_border, dot_color, label_html):
    plain_label = re.sub('<[^>]+>', '', str(label_html))
    return (
        '<span class="task2-pill task2-ui" '
        f'title="{_escape_attr(plain_label)}" '
        f'style="color:{color_text};background:{color_bg};border:1px solid {color_border};">'
        f'<span style="width:8px;height:8px;border-radius:999px;background:{dot_color};display:inline-block;flex:0 0 auto"></span>'
        f'<span class="task2-pill__label">{label_html}</span>'
        '</span>'
    )


def _escape_html(text):
    if text is None:
        return ''
    return (
        str(text)
        .replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
    )


def _update_yaml_indicator(*_):
    info = _inspect_uploaded_yaml(yaml_upload.value)

    if info['state'] == 'valid':
        primary = f"<b>YAML загружен</b>: {_escape_html(info['name'])}"
        if info.get('submission_id'):
            secondary = f" · submission_id: <code>{_escape_html(info['submission_id'])}</code>"
        else:
            secondary = f" · размер: {_escape_html(info.get('size_label') or '—')}"
        yaml_indicator.value = _yaml_indicator_chip(
            '#166534', '#f0fdf4', '#bbf7d0', '#22c55e', primary + secondary
        )
        return

    if info['state'] == 'invalid':
        primary = f"<b>Проверьте YAML</b>: {_escape_html(info['name'] or 'файл')}"
        details = info.get('message') or 'Не удалось быстро проверить файл'
        extra = f" · {_escape_html(details)}"
        if info.get('size_label'):
            extra += f" · размер: {_escape_html(info['size_label'])}"
        yaml_indicator.value = _yaml_indicator_chip(
            '#92400e', '#fffbeb', '#fde68a', '#f59e0b', primary + extra
        )
        return

    yaml_indicator.value = _yaml_indicator_chip(
        '#64748b', '#f8fafc', '#e2e8f0', '#cbd5e1', 'YAML не загружен'
    )



def _refresh_g4f_models(*_, prefer_verified: bool = False):
    current = (g4f_model.value or TASK2_DEFAULT_G4F_MODEL).strip() or TASK2_DEFAULT_G4F_MODEL
    options = _task2_build_g4f_dropdown_options(prefer_verified=prefer_verified, include_auto=True)
    current_values = [value for _, value in options]
    if current not in current_values:
        current = 'auto' if 'auto' in current_values else current_values[0]
    g4f_model.options = options
    g4f_model.value = current

def _task2_preflight_g4f_for_run(model_name):
    selected = (model_name or TASK2_DEFAULT_G4F_MODEL).strip() or TASK2_DEFAULT_G4F_MODEL
    _set_progress(5, 'Проверяю выбранный g4f model/provider', tone='running', details=selected)
    probe = _task2_probe_g4f_model(selected)
    if probe.get('ok'):
        provider_txt = str(probe.get('best_provider') or probe.get('base_provider') or '').strip()
        note = f'g4f preflight ok: {selected}' + (f' · provider: {provider_txt}' if provider_txt else '')
        return {
            'backend': 'g4f',
            'model_id': selected,
            'run_vlm': True,
            'note': note,
            'probe': probe,
        }

    local_choice = (local_model.value or TASK2_DEFAULT_LOCAL_MODEL).strip() or TASK2_DEFAULT_LOCAL_MODEL
    local_ok, local_reason = _local_vlm_runtime_check(local_choice)
    if local_ok:
        note = (
            f'g4f preflight failed ({probe.get("reason") or "unknown"}); '
            f'переключаюсь на local VLM: {local_choice}'
        )
        return {
            'backend': 'qwen3_vl' if 'Qwen3-VL' in local_choice else 'qwen2_vl',
            'model_id': local_choice,
            'run_vlm': True,
            'note': note,
            'probe': probe,
        }

    note = (
        f'g4f preflight failed ({probe.get("reason") or "unknown"}); '
        'продолжаю multimodal pipeline без VLM-captioning.'
    )
    if local_reason:
        note += f' Local runtime reason: {local_reason}'
    return {
        'backend': 'none',
        'model_id': 'auto',
        'run_vlm': False,
        'note': note,
        'probe': probe,
    }


def _render_g4f_probe_status_html(results):
    results = list(results or [])
    total = len(results)
    working = [item for item in results if item.get('ok')]
    relevant = _task2_relevant_g4f_models_from_results(results)
    if not total:
        return (
            '<div class="task2-note task2-ui"><div class="task2-small">'
            'Нажмите <b>«Проверить рабочие g4f модели»</b>, чтобы notebook прогнал все доступные модели '
            'через короткий мультимодальный probe и наполнил выпадающий список только теми, которые реально отвечают в текущей среде.'
            '</div></div>'
        )

    rows = []
    for rank, name in enumerate(relevant[:12], start=1):
        match = next((item for item in working if item.get('model') == name), None)
        provider_txt = _escape_html((match or {}).get('best_provider') or 'best_provider: —')
        mode_txt = _escape_html((match or {}).get('mode') or 'mode: —')
        rows.append(
            '<tr>'
            f'<td style="padding:4px 8px">{rank}</td>'
            f'<td style="padding:4px 8px"><code>{_escape_html(name)}</code></td>'
            f'<td style="padding:4px 8px">{provider_txt}</td>'
            f'<td style="padding:4px 8px">{mode_txt}</td>'
            '</tr>'
        )
    table_html = ''
    if rows:
        table_html = (
            '<table style="border-collapse:collapse;margin-top:8px">'
            '<thead><tr>'
            '<th style="text-align:left;padding:4px 8px">#</th>'
            '<th style="text-align:left;padding:4px 8px">Релевантная рабочая модель</th>'
            '<th style="text-align:left;padding:4px 8px">best_provider</th>'
            '<th style="text-align:left;padding:4px 8px">probe mode</th>'
            '</tr></thead>'
            f'<tbody>{"".join(rows)}</tbody></table>'
        )
    return (
        '<div class="task2-note task2-ui"><div class="task2-small">'
        f'<b>g4f scan завершён.</b> Проверено: <b>{total}</b>; рабочих мультимодальных моделей: <b>{len(working)}</b>; '
        f'выпадающий список обновлён и отсортирован по релевантности для Task 2: <b>{len(relevant)}</b>.'
        f'{table_html}'
        '</div></div>'
    )


def _probe_g4f_models(*_):
    global _TASK2_G4F_SCAN_RESULTS, _TASK2_G4F_LAST_SCAN_META

    if not _g4f_installed():
        g4f_probe_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small"><b>g4f не установлен.</b> '
            'Сначала установите <code>g4f[all]</code>, затем повторите проверку.</div></div>'
        )
        return

    rows = _task2_rank_g4f_rows(_task2_g4f_registry_rows())
    if TASK2_G4F_PROBE_MAX_MODELS > 0:
        rows = rows[:TASK2_G4F_PROBE_MAX_MODELS]

    if not rows:
        g4f_probe_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">'
            'Не удалось получить список моделей из локального реестра g4f.</div></div>'
        )
        return

    scanned = []
    total = len(rows)
    g4f_probe_status.value = (
        '<div class="task2-note task2-ui"><div class="task2-small">'
        f'Запускаю мультимодальный probe по <b>{total}</b> моделям g4f. Это может занять несколько минут.'
        '</div></div>'
    )

    for idx, row in enumerate(rows, start=1):
        name = row.get('name') or ''
        g4f_probe_status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">'
            f'Проверяю <b>{idx}/{total}</b>: <code>{_escape_html(name)}</code> '
            f'(timeout {TASK2_G4F_PROBE_TIMEOUT_SECONDS}s на модель)'
            '</div></div>'
        )
        result = _task2_probe_g4f_model(name, timeout_seconds=TASK2_G4F_PROBE_TIMEOUT_SECONDS)
        result['kind'] = row.get('kind') or 'chat'
        result['base_provider'] = row.get('base_provider') or ''
        result['best_provider'] = row.get('best_provider') or ''
        scanned.append(result)

    _TASK2_G4F_SCAN_RESULTS = scanned
    _TASK2_G4F_LAST_SCAN_META = {
        'checked': len(scanned),
        'working': len([item for item in scanned if item.get('ok')]),
        'timeout_seconds': TASK2_G4F_PROBE_TIMEOUT_SECONDS,
    }
    _refresh_g4f_models(prefer_verified=True)
    g4f_probe_status.value = _render_g4f_probe_status_html(scanned)


def _reset_local_model(*_):
    local_model.options = list_local_vlm_models()
    local_model.value = TASK2_DEFAULT_LOCAL_MODEL


def _update_model_controls(*_):
    use_multimodal = bool(multimodal.value)
    use_g4f = use_multimodal and model_source.value == 'g4f'
    use_local = use_multimodal and model_source.value == 'local'

    model_source.disabled = not use_multimodal
    g4f_model.disabled = not use_g4f
    local_model.disabled = not use_local
    g4f_refresh_btn.disabled = not use_g4f
    local_reset_btn.disabled = not use_local

    g4f_model.layout.display = '' if use_g4f else 'none'
    g4f_refresh_btn.layout.display = '' if use_g4f else 'none'
    g4f_probe_status.layout.display = '' if use_g4f else 'none'
    local_model.layout.display = '' if use_local else 'none'
    local_reset_btn.layout.display = '' if use_local else 'none'

    if not use_multimodal:
        model_hint.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">Multimodal pipeline выключен, поэтому выбор модели для страниц PDF сейчас не используется.</div></div>'
        )
    elif use_g4f:
        if not _g4f_installed():
            model_hint.value = (
                '<div class="task2-note task2-ui"><div class="task2-small"><b>g4f не установлен.</b> '
                'Для запуска без дополнительных действий переключитесь на <code>Local / Transformers</code>. '
                'Если нужен именно g4f, сначала установите пакет <code>pip install -U g4f[all]</code>.</div></div>'
            )
        else:
            model_hint.value = (
                '<div class="task2-note task2-ui"><div class="task2-small">'
                'Кнопка <b>«Проверить рабочие g4f модели»</b> запускает короткий мультимодальный probe по всем моделям из текущего реестра g4f '
                'и затем пересобирает выпадающий список только из рабочих моделей, отсортированных по релевантности для Task 2 '
                '(PDF страницы, figure/table captioning, temporal KG validation). '
                'До проверки список заполняется эвристически по локальному реестру g4f.</div></div>'
            )
            if not _TASK2_G4F_SCAN_RESULTS:
                g4f_probe_status.value = _render_g4f_probe_status_html([])
    else:
        runtime_note = (
            '<b>Local runtime готов.</b> Local VLM будет запускаться через изолированный worker, чтобы не зависеть от состояния текущего notebook-kernel.'
            if TASK2_LOCAL_VLM_RUNTIME_OK else
            '<b>Local runtime пока не подтверждён.</b> Перед запуском notebook попробует автоматически доустановить '             'torch/transformers/qwen-vl-utils и использовать изолированный worker для local VLM.'
        )
        details = f' Текущий статус: <code>{_escape_html(TASK2_LOCAL_VLM_RUNTIME_REASON)}</code>.' if TASK2_LOCAL_VLM_RUNTIME_REASON else ''
        model_hint.value = (
            '<div class="task2-note task2-ui"><div class="task2-small">Для мультимодального шага будет использована локальная Transformers/VLM модель. '
            'По умолчанию выбрана <code>Qwen/Qwen2.5-VL-3B-Instruct</code>, потому что она легче подходит для Google Colab. '
            f'{runtime_note}{details}</div></div>'
        )


yaml_upload.observe(_update_yaml_indicator, names='value')
multimodal.observe(_update_model_controls, names='value')
model_source.observe(_update_model_controls, names='value')


def _g4f_installed():
    try:
        import importlib.util
        return importlib.util.find_spec('g4f') is not None
    except Exception:
        return False

g4f_refresh_btn.on_click(_probe_g4f_models)
local_reset_btn.on_click(_reset_local_model)

_update_yaml_indicator()
_refresh_g4f_models(prefer_verified=False)
_reset_local_model()
_update_model_controls()


def _progress_note_html(text, tone="neutral"):
    palette = {
        "neutral": ("#0f172a", "#f8fafc", "#e2e8f0"),
        "running": ("#1d4ed8", "#eff6ff", "#bfdbfe"),
        "success": ("#166534", "#f0fdf4", "#bbf7d0"),
        "warning": ("#92400e", "#fffbeb", "#fde68a"),
    }
    color_text, color_bg, color_border = palette.get(tone, palette["neutral"])
    return (
        '<div class="task2-note task2-ui" '
        f'style="margin:8px 0;padding:10px 12px;border:1px solid {color_border};background:{color_bg};color:{color_text}">'
        f'<div class="task2-small">{_escape_html(text)}</div></div>'
    )


def _set_progress(percent=0, label='', tone='neutral', details=''):
    percent = max(0, min(100, int(percent or 0)))
    progress_bar.value = percent
    progress_bar.bar_style = 'success' if tone == 'success' else ('warning' if tone == 'warning' else 'info')
    progress_label.value = _progress_note_html(label or 'Ожидание запуска', tone=tone)
    progress_log.value = (
        '<div class="task2-note task2-ui"><div class="task2-small">'
        f'{_escape_html(details)}'
        '</div></div>'
    ) if details else ''


def _task2_progress_callback(payload):
    stage = str(payload.get('stage') or '')
    percent = payload.get('percent') or 0
    message = str(payload.get('message') or stage or 'Выполняется pipeline')

    details_parts = []
    if payload.get('paper_title'):
        details_parts.append(f"paper: {payload.get('paper_title')}")
    elif payload.get('title'):
        details_parts.append(f"paper: {payload.get('title')}")

    if payload.get('item_current') and payload.get('item_total'):
        details_parts.append(f"paper progress: {payload.get('item_current')}/{payload.get('item_total')}")
    if payload.get('page_current') and payload.get('page_total'):
        details_parts.append(f"page progress: {payload.get('page_current')}/{payload.get('page_total')}")

    _set_progress(percent, message, tone='running', details=' · '.join(details_parts))


_set_progress(0, 'Ожидание запуска', tone='neutral')


def run_task2_from_form():
    with out:
        clear_output()
        try:
            if globals().get('TASK2_NOTEBOOK_PIPELINE_MODE') != 'full':
                raise RuntimeError(
                    'Ноутбук не привязан к полноценному Task 2 pipeline. '
                    'Повторно запустите ячейку «Установка и импорт» и убедитесь, что импортируется '
                    'scireason.task2_validation без fallback-ветки.'
                )

            info = _inspect_uploaded_yaml(yaml_upload.value)
            if info['state'] == 'missing':
                status.value = (
                    '<div class="task2-note task2-ui"><div class="task2-small"><b>Нужен YAML-файл.</b> '
                    'Сначала нажмите «Загрузить YAML» и выберите файл .yaml/.yml, '
                    'после этого запустите нижнюю ячейку «Запуск Task 2 из отдельной ячейки».</div></div>'
                )
                display(Markdown("""### Нужно загрузить YAML
1. Нажмите **«Загрузить YAML»**.
2. Выберите файл **Task 1** с расширением `.yaml` или `.yml`.
3. Дождитесь зелёного индикатора **«YAML загружен»**.
4. Если доступен `submission_id`, он сразу появится рядом с названием файла.
5. Снова запустите нижнюю ячейку **«Запуск Task 2 из отдельной ячейки»**."""))
                return

            if info['state'] == 'invalid':
                status.value = (
                    '<div class="task2-note task2-ui"><div class="task2-small"><b>Нужно проверить YAML.</b> '
                    'Файл загружен, но быстрая проверка показала проблему. '
                    'Исправьте YAML и загрузите его заново.</div></div>'
                )
                display(Markdown(f"""### YAML загружен, но не прошёл быструю проверку
- **Файл:** `{info.get('name') or 'unknown'}`
- **Проблема:** {info.get('message') or 'неизвестная ошибка'}
- **Размер:** {info.get('size_label') or '—'}

После исправления снова загрузите файл и дождитесь зелёного индикатора."""))
                return

            status.value = '<div class="task2-note task2-ui"><div class="task2-small"><b>Запуск...</b></div></div>'
            _set_progress(3, 'Инициализирую запуск pipeline', tone='running')
            workdir = Path(tempfile.mkdtemp(prefix='task2_notebook_'))
            trajectory_path = _save_uploaded_yaml(yaml_upload.value, workdir)
            exclusion_path = _save_optional_upload(exclusion_upload.value, workdir, 'task2_exclusions.yaml')
            task1 = load_task1_yaml(trajectory_path)
            display(Markdown(f"""# Загружен YAML
- **topic**: {task1.get('topic')}
- **submission_id**: `{task1.get('submission_id')}`
- **pipeline**: `{TASK2_NOTEBOOK_PIPELINE_MODE}`
- **source**: `{TASK2_NOTEBOOK_PIPELINE_SOURCE}`
- **importance_threshold**: `{float(importance_threshold.value):.2f}`
- **exclusion_yaml**: `{str(exclusion_path) if exclusion_path else '—'}`
"""))

            bundle_kwargs = {
                'progress_callback': _task2_progress_callback,
                'out_dir': Path(out_dir.value),
                'include_auto_pipeline': bool(run_auto.value),
                'multimodal': bool(multimodal.value),
                'run_vlm': bool(multimodal.value),
                'enable_reference_scout': bool(reference_scout.value),
                'enable_remote_lookup': bool(remote_lookup.value),
                'importance_threshold': float(importance_threshold.value),
                'exclusion_spec': exclusion_path,
            }
            if bool(multimodal.value):
                if model_source.value == 'g4f':
                    chosen_g4f_model = (g4f_model.value or TASK2_DEFAULT_G4F_MODEL).strip() or TASK2_DEFAULT_G4F_MODEL
                    g4f_route = _task2_preflight_g4f_for_run(chosen_g4f_model)
                    bundle_kwargs['run_vlm'] = bool(g4f_route.get('run_vlm'))
                    bundle_kwargs['vlm_backend'] = str(g4f_route.get('backend') or 'none')
                    bundle_kwargs['vlm_model_id'] = str(g4f_route.get('model_id') or 'auto')
                    route_note = str(g4f_route.get('note') or '')
                    if route_note:
                        print(f'[info] {route_note}')
                    if bundle_kwargs['vlm_backend'] == 'none':
                        _set_progress(6, 'g4f недоступен — продолжаю без VLM-captioning', tone='warning', details=route_note)
                    elif bundle_kwargs['vlm_backend'] != 'g4f':
                        _set_progress(6, 'g4f недоступен — переключаюсь на local VLM', tone='warning', details=route_note)
                else:
                    chosen_local_model = (local_model.value or TASK2_DEFAULT_LOCAL_MODEL).strip() or TASK2_DEFAULT_LOCAL_MODEL
                    _set_progress(5, 'Проверяю локальный VLM runtime', tone='running', details=chosen_local_model)
                    local_ok, local_reason = _local_vlm_runtime_check(chosen_local_model)
                    if not local_ok:
                        print(f'[info] local VLM runtime preflight failed -> {local_reason}')
                        if not ensure_transformers_stack(chosen_local_model):
                            raise RuntimeError(
                                'Для выбранной local VLM модели не удалось подготовить runtime. '
                                f'Модель: {chosen_local_model}. Причина: {local_reason}. '
                                'Запустите ячейку «Установка и импорт» ещё раз или переключитесь на g4f/отключите multimodal.'
                            )
                        local_ok, local_reason = _local_vlm_runtime_check(chosen_local_model)
                        if not local_ok:
                            raise RuntimeError(
                                'Local VLM runtime после доустановки всё ещё недоступен. '
                                f'Модель: {chosen_local_model}. Причина: {local_reason}. '
                                'Запустите notebook заново после установки зависимостей или выберите g4f.'
                            )
                        globals()['TASK2_LOCAL_VLM_RUNTIME_OK'] = True
                        globals()['TASK2_LOCAL_VLM_RUNTIME_REASON'] = ''
                        _update_model_controls()
                    bundle_kwargs['vlm_backend'] = 'qwen3_vl' if 'Qwen3-VL' in chosen_local_model else 'qwen2_vl'
                    bundle_kwargs['vlm_model_id'] = chosen_local_model

            bundle = build_task2_validation_bundle(
                trajectory_path,
                **bundle_kwargs,
            )
            manifest = _display_manifest(bundle.manifest_path)
            workspace = _build_review_workspace(manifest, task1)
            global _TASK2_LAST_WORKSPACE
            if _TASK2_LAST_WORKSPACE is not None:
                _safe_close_widget_tree(_TASK2_LAST_WORKSPACE)
            _TASK2_LAST_WORKSPACE = workspace
            display(workspace)
            gc.collect()

            _set_progress(100, 'Task 2 bundle собран', tone='success', details=f"bundle: {bundle.bundle_dir}")
            status.value = (
                '<div class="task2-note task2-ui"><div class="task2-small"><b>Готово.</b> '
                'Откройте вкладки для просмотра assertions/графов и выполните разметку в разделе '
                '«Валидация эксперта».</div></div>'
            )
            return bundle
        except ValueError as e:
            _set_progress(progress_bar.value, f'Остановлено: {e}', tone='warning')
            status.value = f'<div class="task2-note task2-ui"><div class="task2-small"><b>Нужно действие:</b> {e}</div></div>'
            display(Markdown("""### Проверьте входной файл
- загрузите один файл `.yaml` или `.yml`;
- дождитесь зелёного индикатора **«YAML загружен»**;
- проверьте, что рядом появился `submission_id` или размер файла;
- убедитесь, что файл не пустой;
- затем запустите нижнюю ячейку **«Запуск Task 2 из отдельной ячейки»** ещё раз."""))
            return None
        except Exception as e:
            _set_progress(progress_bar.value or 0, f'Ошибка: {e}', tone='warning')
            status.value = f'<div class="task2-note task2-ui"><div class="task2-small"><b>Ошибка:</b> {e}</div></div>'
            display(Markdown("""### Во время выполнения возникла ошибка
Ноутбук не упал. Проверьте текст ошибки выше и исправьте входные данные или настройки."""))
            return None


def _on_prepare_run(_):
    info = _inspect_uploaded_yaml(yaml_upload.value)
    if info['state'] == 'missing':
        status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small"><b>Сначала загрузите YAML.</b> '
            'После этого запустите нижнюю ячейку <b>«Запуск Task 2 из отдельной ячейки»</b>.</div></div>'
        )
        _set_progress(0, 'Ожидание YAML', tone='warning')
        return

    if info['state'] == 'invalid':
        status.value = (
            '<div class="task2-note task2-ui"><div class="task2-small"><b>YAML требует исправления.</b> '
            'Загрузите исправленный файл и только потом запускайте нижнюю ячейку.</div></div>'
        )
        _set_progress(progress_bar.value or 0, f"YAML не прошёл проверку: {info.get('message') or 'неизвестная ошибка'}", tone='warning')
        return

    status.value = (
        '<div class="task2-note task2-ui"><div class="task2-small"><b>Форма готова.</b> '
        'Теперь запустите нижнюю ячейку <b>«Запуск Task 2 из отдельной ячейки»</b>. '
        'Именно она держит notebook в состоянии <code>Running</code> во время долгого pipeline.</div></div>'
    )
    _set_progress(max(progress_bar.value, 1), 'Форма проверена — запускайте следующую ячейку', tone='neutral')
    with out:
        clear_output()
        display(Markdown(f"""### Форма готова к запуску
- **YAML:** `{info.get('name') or 'unknown'}`
- **Размер:** {info.get('size_label') or '—'}
- **submission_id:** `{info.get('submission_id') or '—'}`
- **Следующий шаг:** выполните нижнюю ячейку **«Запуск Task 2 из отдельной ячейки»**.
"""))


run_btn.on_click(_on_prepare_run)

display(W.VBox([
    W.HTML(value=_task2_theme_style_html()),
    W.HTML('<h2 class="task2-ui">Task 2 — Validation UI</h2>'),
    W.HTML(
        '<div class="task2-note task2-ui" style="margin-bottom:8px">'
        '<div class="task2-small">Сначала загрузите YAML-файл из Task 1. После быстрой проверки рядом появится зелёный индикатор, '
        'а также <code>submission_id</code> или размер файла. После сборки bundle откроются вкладки просмотра графов '
        'и интерфейс экспертной валидации с экспортом готовых файлов. Если в assertions, карточках или графах текст сокращён, '
        'используйте панель полного текста и инспектор внутри вкладок. Ниже можно выбрать, какую '
        'мультимодальную модель использовать для автоматической генерации графа: g4f или локальную Transformers/VLM. '
        'Онлайн-резолв рекомендуется оставлять включённым, чтобы notebook автоматически находил прямые PDF и строил граф по полным статьям.</div>'
        '</div>'
    ),
    W.HTML(
        '<div class="task2-note task2-ui" style="margin-bottom:8px">'
        '<div class="task2-small"><b>Важно для Colab/Jupyter:</b> долгий Task 2 больше не запускается по кнопке. '
        'Кнопка ниже только проверяет форму. Сам pipeline запускайте следующей code cell — так notebook остаётся '
        'в состоянии <code>Running</code> и не выглядит как idle callback от widgets.</div>'
        '</div>'
    ),
    W.HBox([yaml_upload, yaml_indicator]),
    exclusion_hint,
    W.HBox([exclusion_upload, importance_threshold]),
    out_dir,
    run_auto,
    multimodal,
    reference_scout,
    remote_lookup,
    model_source,
    W.HBox([g4f_model, g4f_refresh_btn]),
    g4f_probe_status,
    W.HBox([local_model, local_reset_btn]),
    model_hint,
    run_btn,
    status,
    progress_label,
    progress_bar,
    progress_log,
    out,
]))


In [ ]:
# @title
# Запуск Task 2 из отдельной ячейки
# Важно: после заполнения формы выше запускайте именно ЭТУ ячейку.
# Так Colab/Jupyter видит активное выполнение и не считает notebook неактивным,
# пока строится Task 2 bundle внутри виджетов и output-панелей.

TASK2_LAST_BUNDLE = run_task2_from_form()


# CLI-режим (тот же workflow без notebook)

Если удобнее запускать всё одной командой из терминала, используйте:

```bash
top-papers-graph task2-bundle   --trajectory path/to/task1.yaml   --out-dir runs/task2_validation   --multimodal   --vlm-backend g4f   --vlm-model-id auto   --suggest-links
```

Для локальной мультимодальной модели:

```bash
top-papers-graph task2-bundle   --trajectory path/to/task1.yaml   --out-dir runs/task2_validation   --multimodal   --vlm-backend qwen2_vl   --vlm-model-id Qwen/Qwen2.5-VL-3B-Instruct   --suggest-links
```

Эквивалентный основной алиас из репозитория:

```bash
top-papers-graph prepare-task2-validation   --trajectory path/to/task1.yaml   --out-dir runs/task2_validation   --multimodal   --vlm-backend qwen2_vl   --vlm-model-id Qwen/Qwen2.5-VL-3B-Instruct   --suggest-links
```


# FAQ

## Что считается эталонным графом?
Эталонный граф строится **только из YAML Task 1** и полностью повторяет шаги эксперта, их evidence и связи между шагами.

## Что считается автоматическим графом?
Автоматический граф строится из **списка статей в YAML** и запускает встроенный конвейер репозитория: статьи → ingestion → temporal KG → review assets → report.

## Что делать, если автоматический граф почти пустой?
Это ожидаемо, если PDF недоступны, в статьях мало структурируемого текста или temporal evidence неявен. В этом случае ориентируйтесь на:
- `reference_scout.json`
- triplets CSV
- `comparison_summary.json`
- review templates из auto pipeline

## Как интерпретировать temporal поля?
- `start_date` / `end_date` — когда связь наблюдается / извлечена;
- `valid_from` / `valid_to` — когда открытие считается валидным;
- `temporal_basis` / `time_source` — откуда взялось время.
